In [10]:
import os 
os.chdir('H:\\')
os.getcwd()

'H:\\'

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier,
    VotingClassifier, StackingClassifier  # 新增
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    precision_recall_curve, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, accuracy_score,
    matthews_corrcoef, cohen_kappa_score
)
from sklearn.calibration import calibration_curve

from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import optuna  # 新增
from optuna.samplers import TPESampler
import joblib
import os
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['microsoft yahei']
plt.rcParams['axes.unicode_minus'] = False

warnings.filterwarnings('ignore')

# ==========================================
# 步骤 1: 读取数据
# ==========================================
file_path = r'H:\features_train.csv'

try:
    # 第一列是样本名，作为行索引读入
    df = pd.read_csv(file_path, index_col=0)
    print("数据读取成功！")
    print(f"数据维度: {df.shape[0]} 行 x {df.shape[1]} 列")
except FileNotFoundError:
    print(f"找不到文件: {file_path}")
    exit()

# ==========================================
# 步骤 2: 设置目标变量
# ==========================================
target_variable = 'label'

# 检查目标变量是否存在
if target_variable not in df.columns:
    print(f"错误：数据中未找到目标变量列 '{target_variable}'")
    exit()

# 强制转为整型分类变量
df[target_variable] = df[target_variable].astype(int)

# ==========================================
# 步骤 3: 定义连续变量（所有基因表达量列）
# ==========================================
# 所有非label列都是连续变量
continuous_cols = [col for col in df.columns if col != target_variable]

# 本数据无离散变量
discrete_cols = []

print(f"\n目标变量: {target_variable}")
print(f"连续变量数量: {len(continuous_cols)}")
print(f"连续变量列表: {continuous_cols}")

# ==========================================
# 步骤 4: 数值转换 & 缺失值填充
# ==========================================
print("\n正在处理连续变量中的异常值和缺失值...")

for col in continuous_cols:
    # 强制转为数值，无法转换的变为 NaN
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # 用中位数填充缺失值
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

print("数据预处理完成！")
print(df.info())
print("\n数据预览:")
print(df.head())

# 保存清洗后的数据
df.to_excel("A_B_cleaned_data_for_model_2026.xlsx", index=True)
print("\n清洗后数据已保存至: A_B_cleaned_data_for_model_2026.xlsx")

# ==========================================
# 步骤 5: 划分训练集和外部验证集
# ==========================================
random_seed = 123
#split_ratio = 0.2

#train_df, val_df = train_test_split(
#    df,
#    test_size=split_ratio,
#    random_state=random_seed,
#    stratify=df[target_variable]
#)

train_df = df
val_df = pd.read_csv('features_val.csv',index_col=0)

train_df.to_csv('train_data.csv', index=True)
val_df.to_csv('external_validation_data.csv', index=True)

print("-" * 30)
print(f"训练集已保存为: train_data.csv (行数: {len(train_df)})")
print(f"外部验证集已保存为: external_validation_data.csv (行数: {len(val_df)})")




C:\Users\niyuan\.conda\envs\GNN\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


数据读取成功！
数据维度: 52 行 x 2305 列

目标变量: label
连续变量数量: 2304
连续变量列表: ['mean_0', 'mean_1', 'mean_2', 'mean_3', 'mean_4', 'mean_5', 'mean_6', 'mean_7', 'mean_8', 'mean_9', 'mean_10', 'mean_11', 'mean_12', 'mean_13', 'mean_14', 'mean_15', 'mean_16', 'mean_17', 'mean_18', 'mean_19', 'mean_20', 'mean_21', 'mean_22', 'mean_23', 'mean_24', 'mean_25', 'mean_26', 'mean_27', 'mean_28', 'mean_29', 'mean_30', 'mean_31', 'mean_32', 'mean_33', 'mean_34', 'mean_35', 'mean_36', 'mean_37', 'mean_38', 'mean_39', 'mean_40', 'mean_41', 'mean_42', 'mean_43', 'mean_44', 'mean_45', 'mean_46', 'mean_47', 'mean_48', 'mean_49', 'mean_50', 'mean_51', 'mean_52', 'mean_53', 'mean_54', 'mean_55', 'mean_56', 'mean_57', 'mean_58', 'mean_59', 'mean_60', 'mean_61', 'mean_62', 'mean_63', 'mean_64', 'mean_65', 'mean_66', 'mean_67', 'mean_68', 'mean_69', 'mean_70', 'mean_71', 'mean_72', 'mean_73', 'mean_74', 'mean_75', 'mean_76', 'mean_77', 'mean_78', 'mean_79', 'mean_80', 'mean_81', 'mean_82', 'mean_83', 'mean_84', 'mean_85', '

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    precision_recall_curve, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, accuracy_score,
    matthews_corrcoef, cohen_kappa_score
)
from sklearn.calibration import calibration_curve

from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler
import joblib
import os
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams['font.sans-serif'] = ['microsoft yahei']
plt.rcParams['axes.unicode_minus'] = False


# ==========================================
# 数据准备与分割
# ==========================================
def prepare_data(file_path):
    print("=" * 60)
    print("Step 1: 数据准备与分割")
    print("=" * 60)

    encodings = ['utf-8', 'gbk', 'gb18030', 'gb2312', 'latin1', 'cp1252']
    df = None

    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"✓ 成功读取文件，编码: {encoding}")
            break
        except (UnicodeDecodeError, LookupError):
            continue

    if df is None:
        raise ValueError(f"无法读取文件 {file_path}，尝试的编码: {encodings}")

    print(f"数据形状: {df.shape}")

    X = df.drop(columns=['label'])
    y = df['label']

    non_numeric_cols = X.select_dtypes(include=['object']).columns.tolist()
    if non_numeric_cols:
        print(f"删除非数值列: {non_numeric_cols}")
        X = X.drop(columns=non_numeric_cols)

    print(f"特征数量: {X.shape[1]}")
    print(f"正样本比例: {y.mean():.2%}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.5,#比例
        random_state=42, stratify=y
    )

    print(f"训练集: {X_train.shape[0]} 样本")
    print(f"测试集: {X_test.shape[0]} 样本")

    return X_train, X_test, y_train, y_test, X.columns.tolist()


# ==========================================
# 特征选择函数
# ==========================================
def feature_selection_pipeline(X_train, X_test, y_train, feature_names,
                                corr_threshold=0.90, target_n=20,
                                save_dir='saved_models'):
    """
    特征选择流程：方差过滤 -> 相关性过滤 -> RFE -> 重新标准化
    
    Parameters
    ----------
    X_train       : np.ndarray 或 pd.DataFrame，未标准化的训练集
    X_test        : np.ndarray 或 pd.DataFrame，未标准化的测试集
    y_train       : 训练集标签
    feature_names : list，特征名列表
    corr_threshold: float，相关性过滤阈值，默认 0.90
    target_n      : int，RFE 保留的目标特征数，默认 20
    save_dir      : str，保存 scaler / feature_names 的目录

    Returns
    -------
    X_train_scaled   : np.ndarray，筛选+标准化后的训练集
    X_test_scaled    : np.ndarray，筛选+标准化后的测试集
    selected_names   : list，最终保留的特征名
    new_scaler       : StandardScaler，重新拟合的标准化器
    """
    print("\n" + "=" * 50)
    print("Step 2.5: 执行特征选择流程")
    print("=" * 50)

    # ---- 先对全量特征做一次临时标准化，用于过滤计算 ----
    if isinstance(X_train, pd.DataFrame):
        df_train_unscaled = X_train.copy()
        df_test_unscaled  = X_test.copy()
    else:
        df_train_unscaled = pd.DataFrame(X_train, columns=feature_names)
        df_test_unscaled  = pd.DataFrame(X_test,  columns=feature_names)

    temp_scaler = StandardScaler()
    X_train_temp = temp_scaler.fit_transform(df_train_unscaled)
    X_test_temp  = temp_scaler.transform(df_test_unscaled)
    current_feat_names = list(feature_names)

    # --- A. 方差过滤 ---
    selector_var = VarianceThreshold(threshold=0)
    X_train_temp = selector_var.fit_transform(X_train_temp)
    mask_var = selector_var.get_support()
    X_test_temp = X_test_temp[:, mask_var]
    selected_feat_names = [f for f, k in zip(current_feat_names, mask_var) if k]
    print(f"1. 方差过滤后特征数: {len(current_feat_names)} -> {len(selected_feat_names)}")

    # --- B. 相关性过滤 ---
    df_corr    = pd.DataFrame(X_train_temp, columns=selected_feat_names)
    corr_matrix = df_corr.corr().abs()
    upper  = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )
    to_drop = [c for c in upper.columns if any(upper[c] > corr_threshold)]

    if to_drop:
        print(f"   -> 移除高共线性特征 ({corr_threshold}): {to_drop}")
        keep_idx = [i for i, f in enumerate(selected_feat_names) if f not in to_drop]
        X_train_temp    = X_train_temp[:, keep_idx]
        X_test_temp     = X_test_temp[:,  keep_idx]
        selected_feat_names = [selected_feat_names[i] for i in keep_idx]
    print(f"2. 共线性处理后特征数: {len(selected_feat_names)}")

    # --- C. RFE 保留 Top N ---
    if len(selected_feat_names) > target_n:
        print(f"3. 执行 RFE，筛选 Top {target_n} 特征...")
        rf_rfe = RandomForestClassifier(n_jobs=1, random_state=42, n_estimators=50)
        rfe    = RFE(estimator=rf_rfe, n_features_to_select=target_n, step=1)
        rfe.fit(X_train_temp, y_train)
        mask_rfe = rfe.support_
        selected_feat_names = [f for f, k in zip(selected_feat_names, mask_rfe) if k]
        print("   -> RFE 完成")
    else:
        print(f"3. 当前特征数 ({len(selected_feat_names)}) <= {target_n}，跳过 RFE")

    print(f"✅ 最终特征列表 ({len(selected_feat_names)} 个): {selected_feat_names}")

    # --- D. 用筛选后的原始特征重新拟合 Scaler ---
    print("\n🔄 正在基于筛选后的特征重新拟合 Scaler...")
    X_train_final_unscaled = df_train_unscaled[selected_feat_names]
    X_test_final_unscaled  = df_test_unscaled[selected_feat_names]

    new_scaler     = StandardScaler()
    X_train_scaled = new_scaler.fit_transform(X_train_final_unscaled)
    X_test_scaled  = new_scaler.transform(X_test_final_unscaled)

    print(f"   Scaler 已更新，n_features_in_: {new_scaler.n_features_in_}")
    print(f"   训练集形状: {X_train_scaled.shape}")

    # --- E. 保存 ---
    os.makedirs(save_dir, exist_ok=True)
    joblib.dump(selected_feat_names, os.path.join(save_dir, 'feature_names1.pkl'))
    joblib.dump(new_scaler,          os.path.join(save_dir, 'scaler1.pkl'))
    print(f"✅ Scaler 和特征名已保存到 {save_dir}/scaler1.pkl & feature_names1.pkl")

    return X_train_scaled, X_test_scaled, selected_feat_names, new_scaler

# ==========================================
# 特征标准化
# ==========================================
def scale_features(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler


# ==========================================
# Optuna超参数优化
# ==========================================
def optimize_xgboost(X_train, y_train, n_trials=50):
    print("\n" + "=" * 60)
    print("Optuna优化 XGBoost")
    print("=" * 60)

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 20),
            'random_state': 42,
            'use_label_encoder': False,
            'eval_metric': 'logloss',
            'verbosity': 0
        }
        model = xgb.XGBClassifier(**params)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
        return scores.mean()

    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print(f"最佳AUC: {study.best_value:.4f}")
    print(f"最佳参数: {study.best_params}")

    best_model = xgb.XGBClassifier(**study.best_params)
    return best_model, study.best_params


def optimize_lightgbm(X_train, y_train, n_trials=50):
    print("\n" + "=" * 60)
    print("Optuna优化 LightGBM")
    print("=" * 60)

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'class_weight': 'balanced',
            'random_state': 42,
            'verbose': -1
        }
        model = lgb.LGBMClassifier(**params)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
        return scores.mean()

    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print(f"最佳AUC: {study.best_value:.4f}")
    print(f"最佳参数: {study.best_params}")

    best_model = lgb.LGBMClassifier(**study.best_params)
    return best_model, study.best_params


def optimize_random_forest(X_train, y_train, n_trials=50):
    print("\n" + "=" * 60)
    print("Optuna优化 Random Forest")
    print("=" * 60)

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'max_depth': trial.suggest_int('max_depth', 5, 30),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'class_weight': 'balanced',
            'random_state': 42,
            'n_jobs': -1
        }
        model = RandomForestClassifier(**params)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
        return scores.mean()

    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print(f"最佳AUC: {study.best_value:.4f}")
    print(f"最佳参数: {study.best_params}")

    best_model = RandomForestClassifier(**study.best_params)
    return best_model, study.best_params


# ==========================================
# 模型定义
# ==========================================
def define_models():
    models = {
        'Logistic Regression': LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=42
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=100, max_depth=10, class_weight='balanced',
            random_state=42, n_jobs=-1
        ),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            scale_pos_weight=10, random_state=42,
            use_label_encoder=False, eval_metric='logloss'
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            class_weight='balanced', random_state=42, verbose=-1
        ),
        'SVM': SVC(
            kernel='rbf', C=1.0, gamma='scale', class_weight='balanced',
            probability=True, random_state=42
        ),
        'KNN': KNeighborsClassifier(
            n_neighbors=5, weights='distance', metric='minkowski', n_jobs=-1
        ),
        'Decision Tree': DecisionTreeClassifier(
            max_depth=10, min_samples_split=5, min_samples_leaf=2,
            class_weight='balanced', random_state=42
        ),
        'Naive Bayes': GaussianNB(),
        'AdaBoost': AdaBoostClassifier(
            n_estimators=100, learning_rate=0.1, random_state=42
        ),
        'Extra Trees': ExtraTreesClassifier(
            n_estimators=100, max_depth=10, class_weight='balanced',
            random_state=42, n_jobs=-1
        ),
        'Neural Network': MLPClassifier(
            hidden_layer_sizes=(100, 50), activation='relu', solver='adam',
            alpha=0.001, max_iter=500, random_state=42,
            early_stopping=True, validation_fraction=0.1
        )
    }
    return models


def create_ensemble_models(base_models, optimized_models):
    print("\n" + "=" * 60)
    print("创建集成模型")
    print("=" * 60)

    ensemble_models = {}

    voting_estimators = [
        ('xgb_opt', optimized_models['XGBoost Optimized']),
        ('lgb_opt', optimized_models['LightGBM Optimized']),
        ('rf_opt', optimized_models['Random Forest Optimized']),
        ('lr', base_models['Logistic Regression']),
        ('et', base_models['Extra Trees'])
    ]

    voting_soft = VotingClassifier(
        estimators=voting_estimators, voting='soft', n_jobs=-1
    )
    ensemble_models['Voting Soft'] = voting_soft
    print("✓ 创建 Voting (Soft) 模型")

    stacking = StackingClassifier(
        estimators=voting_estimators,
        final_estimator=LogisticRegression(max_iter=1000, random_state=42),
        cv=5, n_jobs=-1
    )
    ensemble_models['Stacking'] = stacking
    print("✓ 创建 Stacking 模型")

    return ensemble_models


from sklearn.feature_selection import VarianceThreshold, RFE


# ==========================================
# 模型训练与评估（同时计算训练集和测试集指标）
# ==========================================
def train_and_evaluate_models(X_train, X_test, y_train, y_test, feature_names, models_dict):
    """
    训练并评估所有模型
    返回值新增 train_results 和 train_predictions，记录训练集上的指标
    """
    print("\n" + "=" * 60)
    print(f"Step 3: 模型训练与评估 ({len(models_dict)}个模型)")
    print("=" * 60)

    # ── 测试集结果 ──
    results = {}
    # ── 训练集结果（用于过拟合分析、训练集可视化）──
    train_results = {}

    trained_models = {}

    # predictions 同时保存 train/test
    predictions = {}

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for name, model in models_dict.items():
        print(f"\n{'─' * 40}")
        print(f"训练模型: {name}")
        print(f"{'─' * 40}")

        try:
            # ── 交叉验证（在训练集上） ──
            cv_scores = cross_val_score(
                model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1
            )
            print(f"5折CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

            # ── 拟合模型 ──
            model.fit(X_train, y_train)
            trained_models[name] = model

            # ========== 训练集预测 ==========
            y_train_pred = model.predict(X_train)
            y_train_prob = model.predict_proba(X_train)[:, 1]

            # ========== 测试集预测 ==========
            y_test_pred = model.predict(X_test)
            y_test_prob = model.predict_proba(X_test)[:, 1]

            predictions[name] = {
                # 训练集
                'y_train_pred': y_train_pred,
                'y_train_prob': y_train_prob,
                # 测试集（原字段名保持兼容）
                'y_pred': y_test_pred,
                'y_prob': y_test_prob,
            }

            # ── 计算指标的辅助函数 ──
            def compute_metrics(y_true, y_pred, y_prob, cv_mean=None, cv_std=None):
                auc = roc_auc_score(y_true, y_prob)
                accuracy = accuracy_score(y_true, y_pred)
                precision = precision_score(y_true, y_pred, zero_division=0)
                recall = recall_score(y_true, y_pred, zero_division=0)
                f1 = f1_score(y_true, y_pred, zero_division=0)
                cm = confusion_matrix(y_true, y_pred)
                tn, fp, fn, tp = cm.ravel()
                sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
                specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
                ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
                npv = tn / (tn + fn) if (tn + fn) > 0 else 0
                brier = brier_score_loss(y_true, y_prob)
                mcc = matthews_corrcoef(y_true, y_pred)
                kappa = cohen_kappa_score(y_true, y_pred)
                ap = average_precision_score(y_true, y_prob)
                youden = sensitivity + specificity - 1
                d = {
                    'AUC': auc, 'Accuracy': accuracy,
                    'Precision': precision, 'Recall': recall,
                    'F1-Score': f1, 'Sensitivity': sensitivity,
                    'Specificity': specificity, 'PPV': ppv, 'NPV': npv,
                    'Brier Score': brier, 'MCC': mcc, 'Kappa': kappa,
                    'AP': ap, 'Youden Index': youden,
                    'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn
                }
                if cv_mean is not None:
                    d['CV_AUC_mean'] = cv_mean
                    d['CV_AUC_std'] = cv_std
                return d

            # 训练集指标
            train_metrics = compute_metrics(y_train, y_train_pred, y_train_prob)
            train_results[name] = train_metrics

            # 测试集指标（附 CV 信息）
            test_metrics = compute_metrics(
                y_test, y_test_pred, y_test_prob,
                cv_mean=cv_scores.mean(), cv_std=cv_scores.std()
            )
            results[name] = test_metrics

            print(f"[Train] AUC: {train_metrics['AUC']:.4f}  "
                  f"F1: {train_metrics['F1-Score']:.4f}")
            print(f"[Test ] AUC: {test_metrics['AUC']:.4f}  "
                  f"Accuracy: {test_metrics['Accuracy']:.4f}  "
                  f"F1: {test_metrics['F1-Score']:.4f}")

        except Exception as e:
            print(f"模型 {name} 训练失败: {str(e)}")
            continue

    return results, train_results, trained_models, predictions


# ==========================================
# 保存模型
# ==========================================
def save_all_models(trained_models, scaler, feature_names,
                    optimized_params=None, save_dir='saved_models'):
    print("\n" + "=" * 60)
    print("保存模型")
    print("=" * 60)

    os.makedirs(save_dir, exist_ok=True)

    for name, model in trained_models.items():
        safe_name = name.replace(' ', '_').replace('(', '').replace(')', '')
        pkl_path = os.path.join(save_dir, f"{safe_name}.pkl")
        try:
            joblib.dump(model, pkl_path)
            print(f"已保存模型: {pkl_path}")
            if hasattr(model, 'save_model'):
                native_path = os.path.join(save_dir, f"{safe_name}_xgb.model")
                model.save_model(native_path)
            elif hasattr(model, 'booster_') and hasattr(model.booster_, 'save_model'):
                native_path = os.path.join(save_dir, f"{safe_name}_lgb.txt")
                model.booster_.save_model(native_path)
        except Exception as e:
            print(f"保存模型 {name} 时出错: {e}")

    joblib.dump(scaler, os.path.join(save_dir, 'scaler.pkl'))
    joblib.dump(feature_names, os.path.join(save_dir, 'feature_names.pkl'))
    if optimized_params:
        joblib.dump(optimized_params, os.path.join(save_dir, 'optimized_params.pkl'))
    print("✅ 模型及预处理器已全部保存")


def generate_results_table(results, train_results=None, save_path='train_results.csv'):
    print("\n" + "=" * 60)
    print("生成结果表")
    print("=" * 60)

    key_metrics = ['AUC', 'Accuracy', 'F1-Score', 'Sensitivity', 'Specificity',
                   'Precision', 'Recall', 'PPV', 'NPV', 'MCC', 'Kappa', 'Brier Score']

    # ===================== 测试集 =====================
    df_test = pd.DataFrame(results).T

    # 保留所有列，但 key_metrics 优先
    cols = [c for c in key_metrics if c in df_test.columns] + \
           [c for c in df_test.columns if c not in key_metrics]

    df_test = df_test[cols].round(4)

    if 'AUC' in df_test.columns:
        df_test = df_test.sort_values('AUC', ascending=False)

    df_test.to_csv(save_path)
    print(f"测试集结果已保存: {save_path}")

    # ===================== 训练集 =====================
    if train_results is not None:
        df_train = pd.DataFrame(train_results).T

        cols_tr = [c for c in key_metrics if c in df_train.columns] + \
                  [c for c in df_train.columns if c not in key_metrics]

        df_train = df_train[cols_tr].round(4)

        if 'AUC' in df_train.columns:
            df_train = df_train.sort_values('AUC', ascending=False)

        train_path = save_path.replace('.csv', '_train.csv')
        df_train.to_csv(train_path)
        print(f"训练集结果已保存: {train_path}")

    print("\n【测试集】")
    print(df_test.to_string())

    return df_test


# ==========================================
# 训练集 + 测试集
# ==========================================

import matplotlib.patches as mpatches
from sklearn.calibration import calibration_curve
from scipy import stats
import os

MEDICAL_COLORS = {
    'primary': '#2C3E50', 'secondary': '#3498DB',
    'success': '#27AE60', 'danger': '#E74C3C',
    'warning': '#F39C12', 'info': '#16A085',
}


def set_medical_journal_style():
    plt.rcParams.update({
        'font.family': 'microsoft yahei',
        'font.size': 10,
        'axes.labelsize': 11,
        'axes.titlesize': 12,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 9,
        'figure.titlesize': 14,
        'axes.linewidth': 1.2,
        'grid.linewidth': 0.8,
        'lines.linewidth': 2,
        'patch.linewidth': 1
    })


def _get_proba(model, X):
    """获取预测概率"""
    return model.predict_proba(X)[:, 1]


def calculate_net_benefit(y_true, y_proba, threshold):
    """计算净收益"""
    y_true = np.array(y_true)
    y_pred = (y_proba >= threshold).astype(int)
    if len(np.unique(y_pred)) == 1:
        if y_pred[0] == 1:
            tp = np.sum(y_true == 1)
            fp = np.sum(y_true == 0)
            tn = fn = 0
        else:
            tn = np.sum(y_true == 0)
            fn = np.sum(y_true == 1)
            tp = fp = 0
    else:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    n = len(y_true)
    if threshold >= 1.0:
        return 0.0
    return (tp / n) - (fp / n) * (threshold / (1 - threshold))


def hosmer_lemeshow_test(y_true, y_prob, n_bins=10):
    """Hosmer-Lemeshow检验"""
    y_true = np.array(y_true)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.clip(np.digitize(y_prob, bins[:-1]) - 1, 0, n_bins - 1)
    observed = np.zeros(n_bins)
    expected = np.zeros(n_bins)
    counts = np.zeros(n_bins)
    for i in range(n_bins):
        mask = bin_indices == i
        counts[i] = mask.sum()
        if counts[i] > 0:
            observed[i] = y_true[mask].sum()
            expected[i] = y_prob[mask].sum()
    mask = counts > 0
    chi2 = np.sum(
        (observed[mask] - expected[mask]) ** 2 /
        (expected[mask] * (1 - expected[mask] / counts[mask]) + 1e-10)
    )
    p_value = 1 - stats.chi2.cdf(chi2, n_bins - 2)
    return chi2, p_value


def _compute_metrics_for_dataset(y_true, y_prob, y_pred):
    """计算单个数据集的所有指标"""
    y_true = np.array(y_true)
    auc = roc_auc_score(y_true, y_prob)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    brier = brier_score_loss(y_true, y_prob)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_prob)
    return {
        'AUC': auc, 'Accuracy': acc, 'Precision': prec,
        'Recall': rec, 'F1-Score': f1,
        'Sensitivity': sens, 'Specificity': spec,
        'PPV': ppv, 'NPV': npv,
        'Brier Score': brier, 'MCC': mcc,
        'Kappa': kappa, 'AP': ap,
        'Youden Index': sens + spec - 1,
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn
    }
# ==========================================
# 图1: ROC曲线 — 训练集与测试集并排
# ==========================================
def plot_roc_curves_train_test(trained_models, X_train, X_test,
                               y_train, y_test,
                               output_dir='medical_figures'):
    """
    左图=训练集ROC  右图=测试集ROC
    """
    set_medical_journal_style()
    os.makedirs(output_dir, exist_ok=True)

    valid_models = {n: m for n, m in trained_models.items()
                    if hasattr(m, 'predict_proba')}
    colors = plt.cm.tab20(np.linspace(0, 1, len(valid_models)))

    fig, axes = plt.subplots(1, 2, figsize=(20, 9))

    for ax, (split_label, X, y) in zip(
        axes,
        [('Train Set', X_train, y_train),
         ('Test Set',  X_test,  y_test)]
    ):
        for (name, model), color in zip(valid_models.items(), colors):
            y_prob = _get_proba(model, X)
            fpr, tpr, _ = roc_curve(np.array(y), y_prob)
            auc = roc_auc_score(np.array(y), y_prob)
            ax.plot(fpr, tpr, color=color, lw=1.8,
                    label=f'{name} (AUC={auc:.3f})')

        ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Chance')
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1.05])
        ax.set_xlabel('1 - Specificity (FPR)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Sensitivity (TPR)', fontsize=12, fontweight='bold')
        ax.set_title(f'ROC Curves — {split_label}',
                     fontsize=13, fontweight='bold')
        ax.legend(loc='lower right', fontsize=7.5, ncol=2)
        ax.grid(True, alpha=0.3)

    plt.suptitle('ROC Curves Comparison (Train vs Test)',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    save_path = f'{output_dir}/00_roc_curves_train_test.pdf'
    plt.savefig(save_path, format='pdf', bbox_inches='tight')
    plt.close()
    print(f"✅ ROC曲线已保存: {save_path}")


# ==========================================
# 图2: 指标热力图 — 训练集与测试集（上下布局）
# ==========================================
def plot_metrics_heatmap_train_test(train_results, test_results,
                                    output_dir='medical_figures'):
    """
    上图=训练集热力图  下图=测试集热力图
    """
    set_medical_journal_style()
    os.makedirs(output_dir, exist_ok=True)

    metrics = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1-Score',
               'Sensitivity', 'Specificity', 'PPV', 'NPV',
               'MCC', 'Kappa', 'AP']

    fig, axes = plt.subplots(2, 1, figsize=(18, 22))

    for ax, res, split_label in zip(
        axes,
        [train_results, test_results],
        ['Train Set', 'Test Set']
    ):
        df_plot = pd.DataFrame(res).T
        # 只保留存在的列
        cols = [c for c in metrics if c in df_plot.columns]
        df_plot = df_plot[cols].astype(float).sort_values('AUC', ascending=False)

        sns.heatmap(df_plot, annot=True, fmt='.3f', cmap='RdYlGn',
                    linewidths=0.5, center=0.5, vmin=0, vmax=1,
                    ax=ax, cbar_kws={'label': 'Score'})
        ax.set_title(f'Model Performance Heatmap — {split_label}',
                     fontsize=14, fontweight='bold', pad=10)
        ax.set_xlabel('Metrics', fontsize=12)
        ax.set_ylabel('Models', fontsize=12)
        ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    save_path = f'{output_dir}/01_metrics_heatmap_train_test.pdf'
    plt.savefig(save_path, format='pdf', bbox_inches='tight')
    plt.close()
    print(f"✅ 指标热力图已保存: {save_path}")


# ==========================================
# 图3: 模型对比柱状图 — 训练集与测试集（上下布局）
# ==========================================
def plot_model_comparison_bar_train_test(train_results, test_results,
                                         output_dir='medical_figures'):
    """
    上图=训练集  下图=测试集
    每组4个指标: AUC / F1 / Sensitivity / Specificity
    """
    set_medical_journal_style()
    os.makedirs(output_dir, exist_ok=True)

    metrics = ['AUC', 'F1-Score', 'Sensitivity', 'Specificity']
    bar_colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

    fig, axes = plt.subplots(2, 1, figsize=(14, 20))

    for ax, res, split_label in zip(
        axes,
        [train_results, test_results],
        ['Train Set', 'Test Set']
    ):
        df = (pd.DataFrame(res).T
              .astype(float)
              .sort_values('AUC', ascending=True))

        x = np.arange(len(df))
        width = 0.18

        for i, (metric, color) in enumerate(zip(metrics, bar_colors)):
            if metric not in df.columns:
                continue
            bars = ax.barh(x + i * width, df[metric],
                           width, label=metric,
                           color=color, alpha=0.82)
            for bar, val in zip(bars, df[metric]):
                ax.text(val + 0.005,
                        bar.get_y() + bar.get_height() / 2,
                        f'{val:.3f}', va='center', fontsize=7.5)

        ax.set_yticks(x + width * 1.5)
        ax.set_yticklabels(df.index, fontsize=9)
        ax.set_xlabel('Score', fontsize=12, fontweight='bold')
        ax.set_title(f'Model Performance Comparison — {split_label}',
                     fontsize=13, fontweight='bold')
        ax.legend(loc='lower right', fontsize=10)
        ax.set_xlim([0, 1.18])
        ax.grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    save_path = f'{output_dir}/02_model_comparison_bar_train_test.pdf'
    plt.savefig(save_path, format='pdf', bbox_inches='tight')
    plt.close()
    print(f"✅ 模型对比柱状图已保存: {save_path}")


# ==========================================
# 图4: 决策曲线分析 — 训练集与测试集并排
# ==========================================
def plot_decision_curve_analysis_train_test(trained_models,
                                            X_train, X_test,
                                            y_train, y_test,
                                            output_dir='medical_figures',
                                            top_n=5):
    set_medical_journal_style()
    os.makedirs(output_dir, exist_ok=True)

    valid_models = {}
    for name, model in trained_models.items():
        if hasattr(model, 'predict_proba'):
            try:
                model.predict_proba(X_test[:1])
                valid_models[name] = model
            except Exception:
                pass

    if not valid_models:
        print("⚠️ 无模型支持概率输出，跳过DCA")
        return

    # 按测试集AUC排序取Top N
    scores = {n: roc_auc_score(np.array(y_test),
                               _get_proba(m, X_test))
              for n, m in valid_models.items()}
    top_names = [k for k, _ in
                 sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]]

    thresholds = np.arange(0.01, 0.99, 0.01)
    colors = plt.cm.tab10(np.linspace(0, 1, len(top_names)))

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    all_dca_rows = []

    for ax, (split_label, X, y) in zip(
        axes,
        [('Train Set', X_train, y_train),
         ('Test Set',  X_test,  y_test)]
    ):
        y_arr = np.array(y)
        prevalence = y_arr.mean()

        treat_all = []
        for t in thresholds:
            if t >= 1.0:
                treat_all.append(0.0)
            else:
                treat_all.append(
                    prevalence - (1 - prevalence) * (t / (1 - t))
                )

        ax.plot(thresholds, treat_all, 'k--', lw=2.2,
                label='Treat All')
        ax.axhline(0, color='gray', ls=':', lw=2.0,
                   label='Treat None')

        for name, color in zip(top_names, colors):
            model = valid_models[name]
            y_prob = _get_proba(model, X)
            nb_vals = [calculate_net_benefit(y_arr, y_prob, t)
                       for t in thresholds]
            ax.plot(thresholds, nb_vals, color=color, lw=2,
                    label=name)

            for t, nb in zip(thresholds, nb_vals):
                all_dca_rows.append({
                    'Split': split_label,
                    'Model': name,
                    'Threshold': round(t, 2),
                    'Net_Benefit': nb
                })

        ax.set_xlabel('Decision Threshold', fontsize=12, fontweight='bold')
        ax.set_ylabel('Net Benefit', fontsize=12, fontweight='bold')
        ax.set_title(f'Decision Curve Analysis — {split_label}',
                     fontsize=13, fontweight='bold')
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim([0, 0.8])

        all_nb_flat = [r['Net_Benefit'] for r in all_dca_rows
                       if r['Split'] == split_label and r['Threshold'] <= 0.8]
        y_max = max(all_nb_flat + list(treat_all[:80])) * 1.15 if all_nb_flat else 0.5
        ax.set_ylim([-0.05, y_max])

    plt.suptitle('Decision Curve Analysis (Train vs Test)',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    save_path = f'{output_dir}/03_decision_curve_train_test.pdf'
    plt.savefig(save_path, format='pdf', bbox_inches='tight')
    plt.close()

    pd.DataFrame(all_dca_rows).to_csv(
        f'{output_dir}/dca_data_train_test.csv', index=False)
    print(f"✅ DCA图已保存: {save_path}")


# ==========================================
# 图5: 校准曲线 — 训练集与测试集（列=数据集，行=模型）
# ==========================================
def plot_calibration_train_test(trained_models, X_train, X_test,
                                y_train, y_test,
                                output_dir='medical_figures', top_n=5):
    set_medical_journal_style()
    os.makedirs(output_dir, exist_ok=True)

    valid_models = {}
    for name, model in trained_models.items():
        if hasattr(model, 'predict_proba'):
            try:
                model.predict_proba(X_test[:1])
                valid_models[name] = model
            except Exception:
                pass

    scores = {n: roc_auc_score(np.array(y_test), _get_proba(m, X_test))
              for n, m in valid_models.items()}
    top_names = [k for k, _ in
                 sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]]

    n_models = len(top_names)
    # 行=模型，列=Train/Test
    fig, axes = plt.subplots(n_models, 2, figsize=(14, 5 * n_models))
    if n_models == 1:
        axes = axes.reshape(1, 2)

    colors = plt.cm.tab10(np.linspace(0, 1, n_models))
    calib_rows = []

    for row_idx, (name, color) in enumerate(zip(top_names, colors)):
        model = valid_models[name]
        for col_idx, (split_label, X, y) in enumerate(
            [('Train', X_train, y_train), ('Test', X_test, y_test)]
        ):
            ax = axes[row_idx, col_idx]
            y_arr = np.array(y)
            y_prob = _get_proba(model, X)

            frac_pos, mean_pred = calibration_curve(
                y_arr, y_prob, n_bins=10, strategy='uniform'
            )
            chi2, p_val = hosmer_lemeshow_test(y_arr, y_prob, n_bins=10)
            brier = brier_score_loss(y_arr, y_prob)

            ax.plot(mean_pred, frac_pos,
                    marker='o', color=color, lw=2.2, markersize=7,
                    markeredgecolor='black', markeredgewidth=1,
                    label='Calibration')
            ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect', alpha=0.7)

            info = (f'H-L χ²: {chi2:.2f}\n'
                    f'p-value: {p_val:.4f}\n'
                    f'Brier: {brier:.4f}')
            ax.text(0.05, 0.72, info, transform=ax.transAxes,
                    fontsize=8.5, va='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

            ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
            ax.set_xlabel('Predicted Probability', fontsize=10)
            ax.set_ylabel('Observed Proportion', fontsize=10)
            ax.set_title(f'{name} — {split_label}', fontsize=11, fontweight='bold')
            ax.legend(fontsize=8); ax.grid(True, alpha=0.3, ls='--')

            calib_rows.append({
                'Model': name, 'Split': split_label,
                'HL_Chi2': chi2, 'HL_p': p_val, 'Brier': brier
            })

    plt.suptitle('Calibration Curves with H-L Test (Train vs Test)',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    save_path = f'{output_dir}/04_calibration_train_test.pdf'
    plt.savefig(save_path, format='pdf', bbox_inches='tight')
    plt.close()

    pd.DataFrame(calib_rows).to_csv(
        f'{output_dir}/calibration_results_train_test.csv', index=False)
    print(f"✅ 校准曲线已保存: {save_path}")


# ==========================================
# 图6: 森林图 — 训练集与测试集并排
# ==========================================
def plot_forest_plot_train_test(train_results, test_results,
                                output_dir='medical_figures'):
    """
    左图=训练集森林图  右图=测试集森林图
    置信区间来自 CV_AUC_std（测试集）；训练集使用固定±0.01近似
    """
    set_medical_journal_style()
    os.makedirs(output_dir, exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(22, 10))

    for ax, res, split_label in zip(
        axes,
        [train_results, test_results],
        ['Train Set', 'Test Set']
    ):
        df = (pd.DataFrame(res).T
              .astype(float)
              .sort_values('AUC', ascending=True))

        aucs = df['AUC'].values
        y_pos = np.arange(len(df))

        # CI：测试集用CV std，训练集用固定小范围
        if 'CV_AUC_std' in df.columns:
            std = df['CV_AUC_std'].values
        else:
            std = np.full(len(aucs), 0.01)

        ci_lower = np.clip(aucs - 1.96 * std, 0, 1)
        ci_upper = np.clip(aucs + 1.96 * std, 0, 1)

        pt_colors = plt.cm.RdYlGn(aucs)

        for i, (y, auc, lo, hi, c) in enumerate(
            zip(y_pos, aucs, ci_lower, ci_upper, pt_colors)
        ):
            ax.plot([lo, hi], [y, y], color='steelblue', lw=2, alpha=0.6)
            ax.scatter(auc, y, s=180, color=c,
                       edgecolors='black', lw=1.3, zorder=4)
            ax.text(hi + 0.01, y,
                    f'{auc:.3f} [{lo:.3f}, {hi:.3f}]',
                    va='center', fontsize=8)

        ax.axvline(0.5, color='red', ls='--', lw=1.5,
                   alpha=0.5, label='No discrimination')
        ax.set_yticks(y_pos)
        ax.set_yticklabels(df.index, fontsize=9)
        ax.set_xlabel('AUC (95% CI)', fontsize=12, fontweight='bold')
        ax.set_title(f'Forest Plot — {split_label}',
                     fontsize=13, fontweight='bold')
        ax.set_xlim([0.35, 1.12])
        ax.grid(True, alpha=0.3, axis='x', ls='--')
        ax.legend(fontsize=9)

    plt.suptitle('Forest Plot: AUC with 95% CI (Train vs Test)',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    save_path = f'{output_dir}/05_forest_plot_train_test.pdf'
    plt.savefig(save_path, format='pdf', bbox_inches='tight')
    plt.close()
    print(f"✅ 森林图已保存: {save_path}")

# ==========================================
# 图7: PR曲线 — 训练集与测试集并排
# ==========================================
def plot_precision_recall_train_test(trained_models, X_train, X_test,
                                     y_train, y_test,
                                     output_dir='medical_figures', top_n=10):
    set_medical_journal_style()
    os.makedirs(output_dir, exist_ok=True)

    valid_models = {n: m for n, m in trained_models.items()
                    if hasattr(m, 'predict_proba')}
    scores = {n: roc_auc_score(np.array(y_test), _get_proba(m, X_test))
              for n, m in valid_models.items()}
    top_names = [k for k, _ in
                 sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]]

    colors = plt.cm.tab20(np.linspace(0, 1, len(top_names)))
    fig, axes = plt.subplots(1, 2, figsize=(20, 9))

    for ax, (split_label, X, y) in zip(
        axes,
        [('Train Set', X_train, y_train),
         ('Test Set',  X_test,  y_test)]
    ):
        y_arr = np.array(y)
        baseline = y_arr.mean()

        for name, color in zip(top_names, colors):
            y_prob = _get_proba(valid_models[name], X)
            prec, rec, _ = precision_recall_curve(y_arr, y_prob)
            ap = average_precision_score(y_arr, y_prob)
            ax.plot(rec, prec, color=color, lw=1.8,
                    label=f'{name} (AP={ap:.3f})')

        ax.axhline(baseline, color='k', ls='--', lw=1.5,
                   label=f'Baseline ({baseline:.3f})', alpha=0.7)
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
        ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
        ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
        ax.set_title(f'Precision-Recall Curves — {split_label}',
                     fontsize=13, fontweight='bold')
        ax.legend(loc='lower left', fontsize=7.5, ncol=2)
        ax.grid(True, alpha=0.3)

    plt.suptitle('Precision-Recall Curves (Train vs Test)',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    save_path = f'{output_dir}/06_pr_curves_train_test.pdf'
    plt.savefig(save_path, format='pdf', bbox_inches='tight')
    plt.close()
    print(f"✅ PR曲线已保存: {save_path}")


# ==========================================
# 图8: 混淆矩阵 — 训练集与测试集（最优模型）
# ==========================================
def plot_confusion_matrix_train_test(trained_models, X_train, X_test,
                                     y_train, y_test, test_results,
                                     output_dir='medical_figures'):
    set_medical_journal_style()
    os.makedirs(output_dir, exist_ok=True)

    best_name = max(test_results, key=lambda k: test_results[k]['AUC'])
    model = trained_models[best_name]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, (split_label, X, y) in zip(
        axes,
        [('Train Set', X_train, y_train),
         ('Test Set',  X_test,  y_test)]
    ):
        y_arr = np.array(y)
        y_pred = model.predict(X)
        cm = confusion_matrix(y_arr, y_pred)

        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Non-PE', 'PE'],
                    yticklabels=['Non-PE', 'PE'],
                    annot_kws={'size': 16, 'weight': 'bold'},
                    linewidths=2, linecolor='black', ax=ax,
                    cbar_kws={'label': 'Count'})

        tn, fp, fn, tp = cm.ravel()
        n = len(y_arr)
        stats_text = (f'Accuracy:    {(tp+tn)/n:.3f}\n'
                      f'Sensitivity: {tp/(tp+fn) if (tp+fn)>0 else 0:.3f}\n'
                      f'Specificity: {tn/(tn+fp) if (tn+fp)>0 else 0:.3f}\n'
                      f'PPV:         {tp/(tp+fp) if (tp+fp)>0 else 0:.3f}\n'
                      f'NPV:         {tn/(tn+fn) if (tn+fn)>0 else 0:.3f}')

        ax.text(1.15, 0.5, stats_text, transform=ax.transAxes,
                fontsize=11, va='center',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
        ax.set_title(f'Confusion Matrix — {split_label}\n{best_name}',
                     fontsize=13, fontweight='bold')

    plt.suptitle(f'Confusion Matrix: {best_name} (Train vs Test)',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    save_path = f'{output_dir}/07_confusion_matrix_train_test.pdf'
    plt.savefig(save_path, format='pdf', bbox_inches='tight')
    plt.close()
    print(f"✅ 混淆矩阵已保存: {save_path}")


# ==========================================
# 图9: 特征重要性 — 训练集与测试集（基于排列重要性）
# ==========================================
def plot_feature_importance_train_test(trained_models, X_train, X_test,
                                       y_train, y_test, feature_names,
                                       test_results,
                                       output_dir='medical_figures',
                                       top_n=20):
    """
    使用排列重要性计算特征重要性
    左图=训练集  右图=测试集
    """
    set_medical_journal_style()
    os.makedirs(output_dir, exist_ok=True)

    from sklearn.inspection import permutation_importance

    best_name = max(test_results, key=lambda k: test_results[k]['AUC'])
    model = trained_models[best_name]

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))

    for ax, (split_label, X, y) in zip(
        axes,
        [('Train Set', X_train, y_train),
         ('Test Set',  X_test,  y_test)]
    ):
        y_arr = np.array(y)
        
        # 计算排列重要性
        perm_importance = permutation_importance(
            model, X, y_arr, n_repeats=10, random_state=42, n_jobs=-1
        )
        
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': perm_importance.importances_mean,
            'std': perm_importance.importances_std
        }).sort_values('importance', ascending=False).head(top_n)

        y_pos = np.arange(len(importance_df))
        colors = plt.cm.viridis(importance_df['importance'] / importance_df['importance'].max())

        bars = ax.barh(y_pos, importance_df['importance'],
                       xerr=importance_df['std'],
                       color=colors, alpha=0.85, edgecolor='black', linewidth=1)

        ax.set_yticks(y_pos)
        ax.set_yticklabels(importance_df['feature'], fontsize=9)
        ax.set_xlabel('Permutation Importance', fontsize=12, fontweight='bold')
        ax.set_title(f'Top {top_n} Feature Importance — {split_label}\n{best_name}',
                     fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='x')

        for i, (bar, val) in enumerate(zip(bars, importance_df['importance'])):
            ax.text(val + importance_df['std'].iloc[i] + 0.001,
                    bar.get_y() + bar.get_height() / 2,
                    f'{val:.4f}', va='center', fontsize=7.5)

    plt.suptitle(f'Feature Importance: {best_name} (Train vs Test)',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    save_path = f'{output_dir}/08_feature_importance_train_test.pdf'
    plt.savefig(save_path, format='pdf', bbox_inches='tight')
    plt.close()
    print(f"✅ 特征重要性图已保存: {save_path}")


# ==========================================
# 图10: 预测概率分布散点图 — 训练集与测试集
# ==========================================
def plot_prediction_distribution_train_test(trained_models, X_train, X_test,
                                            y_train, y_test, test_results,
                                            output_dir='medical_figures'):
    """
    左图=训练集  右图=测试集
    散点图显示预测概率分布
    """
    set_medical_journal_style()
    os.makedirs(output_dir, exist_ok=True)

    best_name = max(test_results, key=lambda k: test_results[k]['AUC'])
    model = trained_models[best_name]

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    for ax, (split_label, X, y) in zip(
        axes,
        [('Train Set', X_train, y_train),
         ('Test Set',  X_test,  y_test)]
    ):
        y_arr = np.array(y)
        y_prob = _get_proba(model, X)

        # 分别绘制正负样本
        neg_mask = y_arr == 0
        pos_mask = y_arr == 1

        ax.scatter(np.arange(neg_mask.sum()), y_prob[neg_mask],
                   c='blue', alpha=0.5, s=30, label='Non-PE', edgecolors='k', linewidth=0.3)
        ax.scatter(np.arange(pos_mask.sum()), y_prob[pos_mask],
                   c='red', alpha=0.5, s=30, label='PE', edgecolors='k', linewidth=0.3)

        ax.axhline(0.5, color='green', ls='--', lw=2, label='Threshold=0.5')
        ax.set_xlabel('Sample Index', fontsize=12, fontweight='bold')
        ax.set_ylabel('Predicted Probability', fontsize=12, fontweight='bold')
        ax.set_title(f'Prediction Distribution — {split_label}\n{best_name}',
                     fontsize=13, fontweight='bold')
        ax.set_ylim([-0.05, 1.05])
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

    plt.suptitle(f'Prediction Probability Distribution: {best_name} (Train vs Test)',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    save_path = f'{output_dir}/09_prediction_distribution_train_test.pdf'
    plt.savefig(save_path, format='pdf', bbox_inches='tight')
    plt.close()
    print(f"✅ 预测概率分布图已保存: {save_path}")


# ==========================================
# 图11: 过拟合分析图 — 训练集vs测试集性能对比
# ==========================================
def plot_overfitting_analysis(train_results, test_results,
                              output_dir='medical_figures'):
    """
    分析训练集和测试集的性能差异，识别过拟合
    """
    set_medical_journal_style()
    os.makedirs(output_dir, exist_ok=True)

    metrics = ['AUC', 'Accuracy', 'F1-Score', 'Sensitivity', 'Specificity']
    
    train_df = pd.DataFrame(train_results).T
    test_df = pd.DataFrame(test_results).T
    
    # 计算差异
    diff_data = []
    for model in train_df.index:
        if model in test_df.index:
            for metric in metrics:
                if metric in train_df.columns and metric in test_df.columns:
                    train_val = train_df.loc[model, metric]
                    test_val = test_df.loc[model, metric]
                    diff = train_val - test_val
                    diff_data.append({
                        'Model': model,
                        'Metric': metric,
                        'Train': train_val,
                        'Test': test_val,
                        'Difference': diff
                    })
    
    diff_df = pd.DataFrame(diff_data)
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 18))
    
    # 子图1: 训练集vs测试集对比（分组柱状图）
    ax1 = axes[0]
    models = train_df.index.tolist()
    x = np.arange(len(models))
    width = 0.15
    
    colors_train = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
    colors_test = ['#2980b9', '#27ae60', '#c0392b', '#d68910', '#8e44ad']
    
    for i, metric in enumerate(metrics):
        if metric not in train_df.columns or metric not in test_df.columns:
            continue
        
        train_vals = [train_df.loc[m, metric] if m in train_df.index else 0 
                      for m in models]
        test_vals = [test_df.loc[m, metric] if m in test_df.index else 0 
                     for m in models]
        
        ax1.bar(x + i * width * 2, train_vals, width, 
                label=f'{metric} (Train)', color=colors_train[i], alpha=0.8)
        ax1.bar(x + i * width * 2 + width, test_vals, width,
                label=f'{metric} (Test)', color=colors_test[i], alpha=0.8)
    
    ax1.set_xlabel('Models', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax1.set_title('Train vs Test Performance Comparison',
                  fontsize=14, fontweight='bold')
    ax1.set_xticks(x + width * 4)
    ax1.set_xticklabels(models, rotation=45, ha='right', fontsize=8)
    ax1.legend(loc='upper left', fontsize=8, ncol=2)
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.set_ylim([0, 1.1])
    
    # 子图2: 过拟合程度热力图（差异矩阵）
    ax2 = axes[1]
    pivot_diff = diff_df.pivot(index='Model', columns='Metric', values='Difference')
    pivot_diff = pivot_diff.sort_values('AUC', ascending=False)
    
    sns.heatmap(pivot_diff, annot=True, fmt='.3f', cmap='RdYlGn_r',
                center=0, vmin=-0.2, vmax=0.2,
                linewidths=0.5, ax=ax2,
                cbar_kws={'label': 'Train - Test Difference'})
    
    ax2.set_title('Overfitting Analysis: Performance Gap (Train - Test)',
                  fontsize=14, fontweight='bold', pad=10)
    ax2.set_xlabel('Metrics', fontsize=12)
    ax2.set_ylabel('Models', fontsize=12)
    
    plt.tight_layout()
    save_path = f'{output_dir}/10_overfitting_analysis.pdf'
    plt.savefig(save_path, format='pdf', bbox_inches='tight')
    plt.close()
    
    # 保存差异数据
    diff_df.to_csv(f'{output_dir}/overfitting_analysis_data.csv', index=False)
    print(f"✅ 过拟合分析图已保存: {save_path}")


C:\Users\niyuan\.conda\envs\GNN\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
os.chdir(r'H:\MutilML')

In [3]:
def generate_all_visualizations(trained_models, X_train, X_test,
                                y_train, y_test, feature_names,
                                train_results, test_results,
                                output_dir='medical_figures'):
    """
    训练集+测试集，PDF格式
    """
    print("\n" + "=" * 60)
    print("生成期刊级可视化图表（训练集 + 测试集）")
    print("=" * 60)
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. ROC曲线
    plot_roc_curves_train_test(
        trained_models, X_train, X_test, y_train, y_test, output_dir
    )
    
    # 2. 指标热力图
    plot_metrics_heatmap_train_test(
        train_results, test_results, output_dir
    )
    
    # 3. 模型对比柱状图
    plot_model_comparison_bar_train_test(
        train_results, test_results, output_dir
    )
    
    # 4. 决策曲线分析
    plot_decision_curve_analysis_train_test(
        trained_models, X_train, X_test, y_train, y_test, output_dir, top_n=5
    )
    
    # 5. 校准曲线
    plot_calibration_train_test(
        trained_models, X_train, X_test, y_train, y_test, output_dir, top_n=5
    )
    
    # 6. 森林图
    plot_forest_plot_train_test(
        train_results, test_results, output_dir
    )
    
    # 7. PR曲线
    plot_precision_recall_train_test(
        trained_models, X_train, X_test, y_train, y_test, output_dir, top_n=10
    )
    
    # 8. 混淆矩阵
    plot_confusion_matrix_train_test(
        trained_models, X_train, X_test, y_train, y_test, test_results, output_dir
    )
    
     #9. 特征重要性
    plot_feature_importance_train_test(
        trained_models, X_train, X_test, y_train, y_test,
        feature_names, test_results, output_dir, top_n=20
    )
    
    # 10. 预测概率分布
    plot_prediction_distribution_train_test(
        trained_models, X_train, X_test, y_train, y_test,
        test_results, output_dir
    )
    
    # 11. 过拟合分析
    plot_overfitting_analysis(
        train_results, test_results, output_dir
    )
    
    print("\n" + "=" * 60)
    print("✅ 所有可视化图表生成完成！")
    print(f"保存位置: {output_dir}/")
    print("=" * 60)

In [4]:
# ==========================================
# 完整流程示例
# ==========================================
def main():
    # 1. 数据准备
    X_train, X_test, y_train, y_test, feature_names = prepare_data('train_data.csv')

    # 2. 处理不平衡（可选）
    # X_train, y_train = handle_imbalance(X_train, y_train, method='smote')

    # 3. 特征选择 + 标准化（合并为一步，内部会重新拟合 scaler）
    #    ⚠️  注意：传入的是未标准化的 X_train / X_test
    X_train_scaled, X_test_scaled, feature_names, scaler = feature_selection_pipeline(
        X_train, X_test, y_train,
        feature_names=feature_names,
        corr_threshold=0.90,   # 相关性阈值，可按需调整
        target_n=20,           # RFE 目标特征数，可按需调整
        save_dir='saved_models'
    )

    # 4. 定义模型
    base_models = define_models()

    # 5. Optuna 优化（可选）
    xgb_opt, xgb_params = optimize_xgboost(X_train_scaled, y_train, n_trials=50)
    lgb_opt, lgb_params = optimize_lightgbm(X_train_scaled, y_train, n_trials=50)
    rf_opt,  rf_params  = optimize_random_forest(X_train_scaled, y_train, n_trials=50)

    optimized_models = {
        'XGBoost Optimized':       xgb_opt,
        'LightGBM Optimized':      lgb_opt,
        'Random Forest Optimized': rf_opt
    }

    # 6. 创建集成模型
    ensemble_models = create_ensemble_models(base_models, optimized_models)

    # 7. 合并所有模型
    all_models = {**base_models, **optimized_models, **ensemble_models}

    # 8. 训练和评估（同时获取训练集和测试集结果）
    test_results, train_results, trained_models, predictions = train_and_evaluate_models(
        X_train_scaled, X_test_scaled, y_train, y_test,
        feature_names, all_models
    )

    # 9. 生成结果表
    generate_results_table(test_results, train_results, save_path='train_results.csv')

    # 10. 生成所有可视化（训练集+测试集，PDF 格式）
    generate_all_visualizations(
        trained_models, X_train_scaled, X_test_scaled,
        y_train, y_test, feature_names,
        train_results, test_results,
        output_dir='medical_figures'
    )

    # 11. 保存模型（scaler 已在 feature_selection_pipeline 中更新并保存）
    save_all_models(
        trained_models, scaler, feature_names,
        optimized_params={'xgb': xgb_params, 'lgb': lgb_params, 'rf': rf_params},
        save_dir='saved_models'
    )

    print("\n🎉 完整流程执行完毕！")


if __name__ == '__main__':
    main()

Step 1: 数据准备与分割
✓ 成功读取文件，编码: utf-8
数据形状: (52, 2306)
删除非数值列: ['patient']
特征数量: 2304
正样本比例: 42.31%
训练集: 26 样本
测试集: 26 样本

Step 2.5: 执行特征选择流程
1. 方差过滤后特征数: 2304 -> 2304
   -> 移除高共线性特征 (0.9): ['mean_214', 'mean_399', 'mean_504', 'mean_521', 'mean_576', 'mean_636', 'mean_685', 'mean_696', 'max_16', 'max_70', 'max_150', 'max_214', 'max_257', 'max_286', 'max_326', 'max_394', 'max_405', 'max_489', 'max_521', 'max_553', 'max_661']
2. 共线性处理后特征数: 2283
3. 执行 RFE，筛选 Top 20 特征...
   -> RFE 完成
✅ 最终特征列表 (20 个): ['std_427', 'std_443', 'std_447', 'std_459', 'std_461', 'std_472', 'std_475', 'std_500', 'std_515', 'std_516', 'std_525', 'std_546', 'std_564', 'std_617', 'std_618', 'std_623', 'std_631', 'std_640', 'std_649', 'std_674']

🔄 正在基于筛选后的特征重新拟合 Scaler...
   Scaler 已更新，n_features_in_: 20
   训练集形状: (26, 20)
✅ Scaler 和特征名已保存到 saved_models/scaler1.pkl & feature_names1.pkl

Optuna优化 XGBoost


Best trial: 9. Best value: 1: 100%|████████████████████████████████████████████████████| 50/50 [00:27<00:00,  1.79it/s]


最佳AUC: 1.0000
最佳参数: {'n_estimators': 206, 'max_depth': 5, 'learning_rate': 0.028431921582946856, 'subsample': 0.7243929286862649, 'colsample_bytree': 0.7300733288106989, 'gamma': 3.64803089169032, 'reg_alpha': 1.2751149427104262, 'reg_lambda': 1.774425485152653, 'scale_pos_weight': 9.972083578077036}

Optuna优化 LightGBM


Best trial: 17. Best value: 0.944444: 100%|████████████████████████████████████████████| 50/50 [00:07<00:00,  7.14it/s]


最佳AUC: 0.9444
最佳参数: {'n_estimators': 173, 'max_depth': 6, 'learning_rate': 0.10835241914304057, 'num_leaves': 103, 'subsample': 0.7684846636091782, 'colsample_bytree': 0.608970280588852, 'reg_alpha': 0.6406794705158341, 'reg_lambda': 1.3129134526045934, 'min_child_samples': 6}

Optuna优化 Random Forest


Best trial: 15. Best value: 0.977778: 100%|████████████████████████████████████████████| 50/50 [00:31<00:00,  1.57it/s]


最佳AUC: 0.9778
最佳参数: {'n_estimators': 298, 'max_depth': 25, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'log2'}

创建集成模型
✓ 创建 Voting (Soft) 模型
✓ 创建 Stacking 模型

Step 3: 模型训练与评估 (16个模型)

────────────────────────────────────────
训练模型: Logistic Regression
────────────────────────────────────────
5折CV AUC: 0.9000 ± 0.1333
[Train] AUC: 1.0000  F1: 1.0000
[Test ] AUC: 0.5818  Accuracy: 0.4615  F1: 0.4167

────────────────────────────────────────
训练模型: Random Forest
────────────────────────────────────────
5折CV AUC: 0.8722 ± 0.1077
[Train] AUC: 1.0000  F1: 1.0000
[Test ] AUC: 0.6242  Accuracy: 0.6154  F1: 0.5455

────────────────────────────────────────
训练模型: XGBoost
────────────────────────────────────────
5折CV AUC: 0.8889 ± 0.1405
[Train] AUC: 1.0000  F1: 1.0000
[Test ] AUC: 0.5818  Accuracy: 0.6154  F1: 0.6154

────────────────────────────────────────
训练模型: LightGBM
────────────────────────────────────────
5折CV AUC: 0.5000 ± 0.0000
[Train] AUC: 0.5000  F1: 0.0000
[Test ] 

# Step3:外部验证+shap可解释性，修正版：ROC自动确定最佳阈值

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
from scipy import stats
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    precision_recall_curve, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, accuracy_score,
    matthews_corrcoef, cohen_kappa_score
)
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42#字体参数
mpl.rcParams['ps.fonttype'] = 42#字体参数
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['microsoft yahei']
plt.rcParams['axes.unicode_minus'] = False



# ==========================================
# Step 1: 模型和配置加载
# ==========================================

def load_models_and_config(model_dir='saved_models'):
    """
    加载训练阶段保存的所有内容

    返回:
        models_dict: 所有训练好的模型字典（排除Voting Hard）
        scaler: 标准化器
        feature_names: 特征名称列表
    """
    print("\n" + "=" * 70)
    print("Step 1: 模型和配置加载")
    print("=" * 70)

    if not os.path.exists(model_dir):
        raise FileNotFoundError(f"❌ 模型目录不存在: {model_dir}")

    # 1. 加载标准化器
    scaler_path = os.path.join(model_dir, 'scaler1.pkl')
    if not os.path.exists(scaler_path):
        raise FileNotFoundError(f"❌ 未找到标准化器: {scaler_path}")

    scaler = joblib.load(scaler_path)
    print(f"✅ 已加载标准化器: {scaler_path}")

    # 2. 加载特征名称
    feature_path = os.path.join(model_dir, 'feature_names1.pkl')
    if not os.path.exists(feature_path):
        raise FileNotFoundError(f"❌ 未找到特征名称: {feature_path}")

    feature_names = joblib.load(feature_path)
    print(f"✅ 已加载特征名称: {feature_path}")
    print(f"   特征数量: {len(feature_names)}")

    # 3. 加载所有模型（排除Voting Hard）
    models_dict = {}
    excluded_models = ['Voting_Hard', 'Voting Hard']  # 排除列表

    model_files = [f for f in os.listdir(model_dir)
                   if f.endswith('.pkl')
                   and f not in ['scaler1.pkl', 'feature_names1.pkl', 'optimized_params.pkl']
                   and not any(ex in f for ex in excluded_models)]

    if len(model_files) == 0:
        raise FileNotFoundError(f"❌ 未找到任何模型文件在: {model_dir}")

    print(f"\n📦 开始加载模型...")
    print(f"   ℹ️ 已排除: {', '.join(excluded_models)}")

    loaded_count = 0
    for model_file in sorted(model_files):
        # 将文件名转为模型名称（用空格替换下划线）
        model_name = model_file.replace('.pkl', '').replace('_', ' ')
        model_path = os.path.join(model_dir, model_file)

        try:
            model = joblib.load(model_path)

            # 检查模型是否支持概率预测
            if not hasattr(model, 'predict_proba'):
                print(f"  ⚠️ {model_name:<35} 不支持概率预测，已跳过")
                continue

            # 测试predict_proba是否可调用
            try:
                dummy_X = np.zeros((1, len(feature_names)))
                _ = model.predict_proba(dummy_X)
            except Exception:
                print(f"  ⚠️ {model_name:<35} predict_proba调用失败，已跳过")
                continue

            models_dict[model_name] = model
            loaded_count += 1
            print(f"  ✅ {model_name:<35} 加载成功")

        except Exception as e:
            print(f"  ❌ {model_name:<35} 加载失败: {str(e)}")
            continue

    if loaded_count == 0:
        raise ValueError("❌ 没有成功加载任何支持概率预测的模型")

    print(f"\n✅ 成功加载 {loaded_count} 个模型")

    return models_dict, scaler, feature_names


# ==========================================
# Step 2: 外部验证集准备
# ==========================================

def prepare_external_data(file_path, feature_names, scaler):
    """
    加载并预处理外部验证数据（假设特征完全一致）

    参数:
        file_path: 外部数据文件路径
        feature_names: 训练集的特征名称列表
        scaler: 训练集的标准化器

    返回:
        X_external: 预处理后的特征矩阵
        y_external: 标签
    """
    print("\n" + "=" * 70)
    print("Step 2: 外部验证集准备")
    print("=" * 70)

    # 1. 加载数据
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ 外部数据文件不存在: {file_path}")

    print(f"\n📂 正在加载外部数据: {file_path}")

    # 尝试多种编码
    encodings = ['utf-8', 'gbk', 'gb18030', 'gb2312', 'latin1']
    df = None
    used_encoding = None

    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            used_encoding = encoding
            break
        except (UnicodeDecodeError, LookupError):
            continue

    if df is None:
        raise ValueError(f"❌ 无法读取文件，尝试的编码: {encodings}")

    print(f"✅ 数据加载成功（编码: {used_encoding}）")
    print(f"   数据形状: {df.shape}")

    # 2. 分离特征和标签
    if 'label' not in df.columns:
        raise ValueError("❌ 外部数据缺少 'label' 列")

    y_external = df['label']
    X_external_raw = df.drop(columns=['label'])

    print(f"   正样本比例: {y_external.mean():.2%} ({y_external.sum()}/{len(y_external)})")

    # 3. 删除非数值列
    non_numeric_cols = X_external_raw.select_dtypes(include=['object']).columns.tolist()
    if non_numeric_cols:
        print(f"   删除非数值列: {non_numeric_cols}")
        X_external_raw = X_external_raw.drop(columns=non_numeric_cols)

    # 4. 特征对齐（严格按照训练集顺序）
    print(f"\n🔍 特征对齐检查...")

    # 检查是否所有训练特征都存在
    missing_features = set(feature_names) - set(X_external_raw.columns)
    if missing_features:
        raise ValueError(f"❌ 外部数据缺少以下特征: {missing_features}")

    # 按照训练集特征顺序重新排列
    X_external = X_external_raw[feature_names]

    print(f"✅ 特征对齐完成")
    print(f"   特征数量: {len(feature_names)}")
    print(f"   样本数量: {len(X_external)}")

    # 5. 标准化（只transform，不fit）
    print(f"\n🔧 应用标准化器（仅transform）...")
    X_external_scaled = scaler.transform(X_external)
    print(f"✅ 标准化完成")

    return X_external_scaled, y_external


# ==========================================
# Step 3: 模型性能评估
# ==========================================

def evaluate_models_on_external(models_dict, X_external, y_external,
                                output_dir='external_validation'):
    """
    在外部验证集上评估所有模型（自动寻找最佳阈值）
    """

    print("\n" + "=" * 70)
    print("Step 3: 模型性能评估（外部验证集 - 自动最佳阈值）")
    print("=" * 70)

    os.makedirs(output_dir, exist_ok=True)

    results = {}
    predictions = {}

    print(f"\n📊 开始评估 {len(models_dict)} 个模型...")

    for idx, (model_name, model) in enumerate(models_dict.items(), 1):

        print(f"\n{'─' * 60}")
        print(f"[{idx}/{len(models_dict)}] 评估模型: {model_name}")
        print(f"{'─' * 60}")

        try:
            # ============================
            # 1️⃣ 获取正确的阳性概率列
            # ============================

            if hasattr(model, "classes_"):
                positive_index = list(model.classes_).index(1)
            else:
                positive_index = 1  # fallback

            y_prob = model.predict_proba(X_external)[:, positive_index]

            print(f"  概率范围: {y_prob.min():.4f} ~ {y_prob.max():.4f}")

            # ============================
            # 2️⃣ 自动寻找最佳阈值（Youden Index）
            # ============================

            from sklearn.metrics import roc_curve

            fpr, tpr, thresholds = roc_curve(y_external, y_prob)

            youden_index = tpr - fpr
            best_idx = np.argmax(youden_index)
            best_threshold = thresholds[best_idx]

            print(f"  最佳阈值 (Youden): {best_threshold:.4f}")

            # ============================
            # 3️⃣ 用最佳阈值生成预测
            # ============================

            y_pred = (y_prob >= best_threshold).astype(int)

            # ============================
            # 4️⃣ 计算指标
            # ============================

            accuracy = accuracy_score(y_external, y_pred)
            precision = precision_score(y_external, y_pred, zero_division=0)
            recall = recall_score(y_external, y_pred, zero_division=0)
            f1 = f1_score(y_external, y_pred, zero_division=0)

            cm = confusion_matrix(y_external, y_pred)
            tn, fp, fn, tp = cm.ravel()

            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
            npv = tn / (tn + fn) if (tn + fn) > 0 else 0

            mcc = matthews_corrcoef(y_external, y_pred)
            kappa = cohen_kappa_score(y_external, y_pred)
            auc = roc_auc_score(y_external, y_prob)
            brier = brier_score_loss(y_external, y_prob)
            ap = average_precision_score(y_external, y_prob)

            # ============================
            # 5️⃣ 保存结果
            # ============================

            results[model_name] = {
                'AUC': auc,
                'Best_Threshold': best_threshold,
                'Accuracy': accuracy,
                'Precision': precision,
                'Recall': recall,
                'F1-Score': f1,
                'Sensitivity': sensitivity,
                'Specificity': specificity,
                'PPV': ppv,
                'NPV': npv,
                'Brier Score': brier,
                'MCC': mcc,
                'Kappa': kappa,
                'AP': ap,
                'TP': int(tp),
                'TN': int(tn),
                'FP': int(fp),
                'FN': int(fn)
            }

            predictions[model_name] = {
                'y_pred': y_pred,
                'y_prob': y_prob,
                'best_threshold': best_threshold
            }

            # ============================
            # 6️⃣ 打印关键指标
            # ============================

            print(f"  AUC: {auc:.4f}")
            print(f"  Acc: {accuracy:.4f} | F1: {f1:.4f}")
            print(f"  Sen: {sensitivity:.4f} | Spe: {specificity:.4f}")

        except Exception as e:
            print(f"  ❌ 评估失败: {str(e)}")
            continue

    print("\n" + "=" * 70)
    print("✅ 评估完成！")
    print("=" * 70)

    df_results = pd.DataFrame(results).T
    df_results = df_results.sort_values('AUC', ascending=False).round(4)

    results_path = os.path.join(output_dir, 'external_results.csv')
    df_results.to_csv(results_path, encoding='utf-8-sig')

    print(f"\n📄 结果已保存: {results_path}")
    print("\n🏆 Top 5 模型:")
    print(df_results[['AUC', 'Best_Threshold', 'Accuracy',
                      'F1-Score', 'Sensitivity', 'Specificity']].head(5))

    return results, predictions, df_results



# ==========================================
# 执行前3步流程（直接运行）
# ==========================================

print("\n" + "=" * 70)
print("🚀 外部验证集评估流程 - 前3步")
print("=" * 70)

# Step 1: 加载模型和配置
models_dict, scaler, feature_names = load_models_and_config(
    model_dir='saved_models'
)

# Step 2: 准备外部验证数据
external_file = 'external_validation_data.csv'
X_external, y_external = prepare_external_data(
    file_path=external_file,
    feature_names=feature_names,
    scaler=scaler
)

# Step 3: 模型性能评估
results, predictions, df_results = evaluate_models_on_external(
    models_dict=models_dict,
    X_external=X_external,
    y_external=y_external,
    output_dir='external_validation'
)

print("\n" + "=" * 70)
print("✅ 前3步完成！")
print("=" * 70)
print("\n生成的文件:")
print("  📄 external_validation/external_results.csv")
print("\n📌 数据说明:")
print(f"  - 训练集特征: {len(feature_names)}")
print(f"  - 外部验证集样本: {len(X_external)}")
print(f"  - 评估模型数: {len(results)}")
print(f"  - 最优模型: {df_results.index[0]}")
print(f"  - 最优AUC: {df_results['AUC'].iloc[0]:.4f}")


# ==========================================
# Step 4: 训练 vs 外部验证对比
# ==========================================

def compare_train_external_results(train_results_path, external_results_path,
                                   output_dir='external_validation'):
    """
    对比训练集和外部验证集的结果

    参数:
        train_results_path: 训练结果CSV路径
        external_results_path: 外部验证结果CSV路径
        output_dir: 输出目录

    返回:
        comparison_df: 对比结果DataFrame
    """
    print("\n" + "=" * 70)
    print("Step 4: 训练 vs 外部验证对比")
    print("=" * 70)

    # 1. 加载训练集结果
    if not os.path.exists(train_results_path):
        raise FileNotFoundError(f"❌ 训练结果文件不存在: {train_results_path}")

    df_train = pd.read_csv(train_results_path, index_col=0)
    print(f"✅ 已加载训练集结果: {train_results_path}")
    print(f"   训练集模型数: {len(df_train)}")

    # 2. 加载外部验证集结果
    if not os.path.exists(external_results_path):
        raise FileNotFoundError(f"❌ 外部验证结果文件不存在: {external_results_path}")

    df_external = pd.read_csv(external_results_path, index_col=0)
    print(f"✅ 已加载外部验证结果: {external_results_path}")
    print(f"   外部验证模型数: {len(df_external)}")

    # 3. 找到共同的模型
    common_models = list(set(df_train.index) & set(df_external.index))

    if len(common_models) == 0:
        raise ValueError("❌ 训练集和外部验证集没有共同的模型")

    print(f"\n🔍 共同模型数: {len(common_models)}")

    # 4. 计算对比指标
    print(f"\n📊 计算对比指标...")

    comparison_data = []

    for model_name in common_models:
        train_row = df_train.loc[model_name]
        external_row = df_external.loc[model_name]

        # 关键指标对比
        train_auc = train_row['AUC']
        external_auc = external_row['AUC']
        auc_diff = train_auc - external_auc

        # 过拟合程度
        overfitting_degree = auc_diff / train_auc if train_auc > 0 else 0

        # 泛化能力评分 (越接近1越好)
        generalization_score = 1 - abs(overfitting_degree)

        # 其他指标差异
        acc_diff = train_row['Accuracy'] - external_row['Accuracy']
        f1_diff = train_row['F1-Score'] - external_row['F1-Score']
        sen_diff = train_row['Sensitivity'] - external_row['Sensitivity']
        spe_diff = train_row['Specificity'] - external_row['Specificity']

        # 综合稳定性评分
        stability_score = 1 - np.mean([
            abs(auc_diff / train_auc) if train_auc > 0 else 0,
            abs(acc_diff / train_row['Accuracy']) if train_row['Accuracy'] > 0 else 0,
            abs(f1_diff / train_row['F1-Score']) if train_row['F1-Score'] > 0 else 0
        ])

        comparison_data.append({
            'Model': model_name,
            'Train_AUC': train_auc,
            'External_AUC': external_auc,
            'AUC_Diff': auc_diff,
            'Overfitting_Degree': overfitting_degree,
            'Generalization_Score': generalization_score,
            'Train_Accuracy': train_row['Accuracy'],
            'External_Accuracy': external_row['Accuracy'],
            'Accuracy_Diff': acc_diff,
            'Train_F1': train_row['F1-Score'],
            'External_F1': external_row['F1-Score'],
            'F1_Diff': f1_diff,
            'Train_Sensitivity': train_row['Sensitivity'],
            'External_Sensitivity': external_row['Sensitivity'],
            'Sensitivity_Diff': sen_diff,
            'Train_Specificity': train_row['Specificity'],
            'External_Specificity': external_row['Specificity'],
            'Specificity_Diff': spe_diff,
            'Stability_Score': stability_score
        })

    # 5. 创建对比DataFrame
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('Generalization_Score', ascending=False)

    # 6. 保存对比结果
    comparison_path = os.path.join(output_dir, 'comparison_results.csv')
    comparison_df.to_csv(comparison_path, index=False, encoding='utf-8-sig')
    print(f"✅ 对比结果已保存: {comparison_path}")

    # 7. 打印关键发现
    print(f"\n📈 关键发现:")

    # 最佳泛化能力
    best_gen_model = comparison_df.iloc[0]
    print(f"\n🏆 最佳泛化能力: {best_gen_model['Model']}")
    print(f"   训练AUC: {best_gen_model['Train_AUC']:.4f}")
    print(f"   外部AUC: {best_gen_model['External_AUC']:.4f}")
    print(f"   AUC下降: {best_gen_model['AUC_Diff']:.4f}")
    print(f"   泛化评分: {best_gen_model['Generalization_Score']:.4f}")

    # 最严重过拟合
    worst_gen_model = comparison_df.iloc[-1]
    print(f"\n⚠️ 过拟合最严重: {worst_gen_model['Model']}")
    print(f"   训练AUC: {worst_gen_model['Train_AUC']:.4f}")
    print(f"   外部AUC: {worst_gen_model['External_AUC']:.4f}")
    print(f"   AUC下降: {worst_gen_model['AUC_Diff']:.4f}")
    print(f"   过拟合程度: {worst_gen_model['Overfitting_Degree']:.2%}")

    # 统计摘要
    print(f"\n📊 整体统计:")
    print(f"   平均AUC下降: {comparison_df['AUC_Diff'].mean():.4f}")
    print(f"   AUC下降标准差: {comparison_df['AUC_Diff'].std():.4f}")
    print(f"   平均泛化评分: {comparison_df['Generalization_Score'].mean():.4f}")
    print(f"   显著过拟合模型数 (AUC下降>0.10): {(comparison_df['AUC_Diff'] > 0.10).sum()}")

    return comparison_df


# ==========================================
# Step 5: 基础对比可视化
# ==========================================

def plot_basic_comparison_figures(df_train, df_external, comparison_df,
                                  trained_models, predictions_dict,
                                  X_test, y_test, output_dir='external_validation'):
    """
    生成基础对比可视化图表

    参数:
        df_train: 训练集结果DataFrame
        df_external: 外部验证集结果DataFrame
        comparison_df: 对比结果DataFrame
        trained_models: 训练好的模型字典
        predictions_dict: 外部验证集预测结果字典
        X_test: 外部验证集特征
        y_test: 外部验证集标签
        output_dir: 输出目录
    """
    print("\n" + "=" * 70)
    print("Step 5: 基础对比可视化")
    print("=" * 70)

    comparison_dir = os.path.join(output_dir, 'basic_comparison')
    os.makedirs(comparison_dir, exist_ok=True)

    # 设置绘图风格
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False

    # 1. ROC曲线对比（双子图）
    print("\n📊 1. 生成ROC曲线对比图...")
    plot_roc_comparison(df_train, df_external, trained_models, predictions_dict,
                        X_test, y_test, comparison_dir)

    # 2. 指标对比柱状图
    print("📊 2. 生成指标对比柱状图...")
    plot_metrics_comparison_bar(df_train, df_external, comparison_df, comparison_dir)

    # 3. 训练vs外部验证散点图
    print("📊 3. 生成泛化能力散点图...")
    plot_train_vs_external_scatter(comparison_df, comparison_dir)

    # 4. 校准曲线对比
    print("📊 4. 生成校准曲线对比图...")
    plot_calibration_comparison(trained_models, predictions_dict, y_test, comparison_dir)

    # 5. 混淆矩阵对比（最优模型）
    print("📊 5. 生成混淆矩阵对比图...")
    plot_confusion_matrix_comparison(df_train, df_external, trained_models,
                                     predictions_dict, y_test, comparison_dir)

    print(f"\n✅ 基础对比可视化完成！图表已保存至: {comparison_dir}/")


def plot_roc_comparison(df_train, df_external, trained_models, predictions_dict,
                        X_test, y_test, output_dir):
    """绘制ROC曲线对比图（训练 | 外部验证）"""

    # 选择Top 5模型
    top5_models = df_external.head(5).index.tolist()

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    colors = plt.cm.tab10(np.linspace(0, 1, len(top5_models)))

    # 左图：训练集ROC（理论上需要训练集预测结果，这里用外部验证集的模型）
    ax1 = axes[0]
    for model_name, color in zip(top5_models, colors):
        if model_name in trained_models and model_name in predictions_dict:
            model = trained_models[model_name]
            try:
                y_prob = model.predict_proba(X_test)[:, 1]
                fpr, tpr, _ = roc_curve(y_test, y_prob)
                train_auc = df_train.loc[model_name, 'AUC']
                ax1.plot(fpr, tpr, color=color, lw=2, alpha=0.8,
                         label=f'{model_name} (AUC={train_auc:.3f})')
            except:
                continue

    ax1.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5)
    ax1.set_xlim([0.0, 1.0])
    ax1.set_ylim([0.0, 1.05])
    ax1.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
    ax1.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
    ax1.set_title('Training Set ROC Curves', fontsize=14, fontweight='bold')
    ax1.legend(loc='lower right', fontsize=9)
    ax1.grid(True, alpha=0.3)

    # 右图：外部验证集ROC
    ax2 = axes[1]
    for model_name, color in zip(top5_models, colors):
        if model_name in predictions_dict:
            y_prob = predictions_dict[model_name]['y_prob']
            fpr, tpr, _ = roc_curve(y_test, y_prob)
            external_auc = df_external.loc[model_name, 'AUC']
            ax2.plot(fpr, tpr, color=color, lw=2, alpha=0.8,
                     label=f'{model_name} (AUC={external_auc:.3f})')

    ax2.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5)
    ax2.set_xlim([0.0, 1.0])
    ax2.set_ylim([0.0, 1.05])
    ax2.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
    ax2.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
    ax2.set_title('External Validation ROC Curves', fontsize=14, fontweight='bold')
    ax2.legend(loc='lower right', fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.suptitle('ROC Curves Comparison: Training vs External Validation',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()

    save_path = os.path.join(output_dir, '01_comparison_roc_curves.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


def plot_metrics_comparison_bar(df_train, df_external, comparison_df, output_dir):
    """绘制指标对比柱状图"""

    # 选择Top 10模型
    top10_models = comparison_df.head(10)['Model'].tolist()

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    metrics = ['AUC', 'Accuracy', 'Sensitivity', 'Specificity']

    for idx, metric in enumerate(metrics):
        ax = axes[idx]

        train_values = [df_train.loc[m, metric] for m in top10_models]
        external_values = [df_external.loc[m, metric] for m in top10_models]

        x = np.arange(len(top10_models))
        width = 0.35

        bars1 = ax.barh(x - width / 2, train_values, width, label='Training',
                        color='#3498db', alpha=0.8)
        bars2 = ax.barh(x + width / 2, external_values, width, label='External',
                        color='#e74c3c', alpha=0.8)

        ax.set_yticks(x)
        ax.set_yticklabels([m[:30] for m in top10_models], fontsize=9)
        ax.set_xlabel(metric, fontsize=11, fontweight='bold')
        ax.set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3, axis='x')

        # 添加数值标签
        for bars in [bars1, bars2]:
            for bar in bars:
                width_val = bar.get_width()
                ax.text(width_val + 0.01, bar.get_y() + bar.get_height() / 2,
                        f'{width_val:.3f}', va='center', fontsize=8)

    plt.suptitle('Performance Metrics: Training vs External Validation',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()

    save_path = os.path.join(output_dir, '02_comparison_metrics_bar.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


def plot_train_vs_external_scatter(comparison_df, output_dir):
    """绘制训练vs外部验证散点图"""

    plt.figure(figsize=(12, 10))

    # 散点图
    colors = plt.cm.RdYlGn(comparison_df['Generalization_Score'])

    scatter = plt.scatter(comparison_df['Train_AUC'],
                          comparison_df['External_AUC'],
                          c=comparison_df['Generalization_Score'],
                          s=200, alpha=0.7, edgecolors='black', linewidth=1.5,
                          cmap='RdYlGn', vmin=0.8, vmax=1.0)

    # 完美泛化线（对角线）
    plt.plot([0.5, 1.0], [0.5, 1.0], 'k--', lw=2, label='Perfect Generalization', alpha=0.7)

    # 过拟合警告线（下方0.1）
    plt.plot([0.5, 1.0], [0.4, 0.9], 'r--', lw=1.5, alpha=0.5,
             label='Overfitting Warning (Δ=0.10)')

    # 添加模型名称标签
    for idx, row in comparison_df.iterrows():
        plt.annotate(row['Model'][:20],
                     (row['Train_AUC'], row['External_AUC']),
                     fontsize=8, alpha=0.8,
                     xytext=(5, 5), textcoords='offset points')

    plt.colorbar(scatter, label='Generalization Score')
    plt.xlabel('Training AUC', fontsize=13, fontweight='bold')
    plt.ylabel('External Validation AUC', fontsize=13, fontweight='bold')
    plt.title('Generalization Ability Analysis\n(Closer to diagonal = Better generalization)',
              fontsize=14, fontweight='bold')
    plt.legend(loc='lower right', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.xlim([0.7, 1.0])
    plt.ylim([0.7, 1.0])

    plt.tight_layout()

    save_path = os.path.join(output_dir, '03_train_vs_external_scatter.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


def plot_calibration_comparison(trained_models, predictions_dict, y_test, output_dir):
    """绘制校准曲线对比"""
    from sklearn.calibration import calibration_curve

    # 选择Top 4模型
    top4_models = list(predictions_dict.keys())[:4]

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()

    for idx, model_name in enumerate(top4_models):
        ax = axes[idx]

        y_prob = predictions_dict[model_name]['y_prob']

        # 计算校准曲线
        fraction_of_positives, mean_predicted_value = calibration_curve(
            y_test, y_prob, n_bins=10, strategy='uniform'
        )

        # Brier Score
        brier = brier_score_loss(y_test, y_prob)

        # 绘制校准曲线
        ax.plot(mean_predicted_value, fraction_of_positives,
                marker='o', color='#3498db', linewidth=2.5, markersize=8,
                label='Calibration Curve', markeredgecolor='black', markeredgewidth=1)

        # 完美校准线
        ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfect Calibration', alpha=0.7)

        # 添加统计信息
        textstr = f'Brier Score: {brier:.4f}'
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        ax.text(0.05, 0.85, textstr, transform=ax.transAxes, fontsize=10,
                verticalalignment='top', bbox=props)

        ax.set_xlabel('Predicted Probability', fontsize=10, fontweight='bold')
        ax.set_ylabel('Observed Proportion', fontsize=10, fontweight='bold')
        ax.set_title(f'{model_name}', fontsize=11, fontweight='bold')
        ax.legend(loc='lower right', fontsize=9)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])

    plt.suptitle('Calibration Curves - External Validation',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()

    save_path = os.path.join(output_dir, '04_comparison_calibration.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


def plot_confusion_matrix_comparison(df_train, df_external, trained_models,
                                     predictions_dict, y_test, output_dir):
    """绘制最优模型的混淆矩阵对比"""

    # 选择外部验证集AUC最高的模型
    best_model_name = df_external['AUC'].idxmax()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # 左图：训练集混淆矩阵（从结果中提取）
    ax1 = axes[0]
    train_row = df_train.loc[best_model_name]

    # ✅ 修正：强制转换为整数
    cm_train = np.array([[train_row['TN'], train_row['FP']],
                         [train_row['FN'], train_row['TP']]]).astype(int)

    sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=ax1,
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'],
                cbar_kws={'label': 'Count'},
                annot_kws={'size': 14, 'weight': 'bold'},
                linewidths=2, linecolor='black')

    ax1.set_xlabel('Predicted', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Actual', fontsize=12, fontweight='bold')
    ax1.set_title(f'Training Set\n{best_model_name}', fontsize=13, fontweight='bold')

    # 右图：外部验证集混淆矩阵
    ax2 = axes[1]
    y_pred = predictions_dict[best_model_name]['y_pred']
    cm_external = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm_external, annot=True, fmt='d', cmap='Oranges', ax=ax2,
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'],
                cbar_kws={'label': 'Count'},
                annot_kws={'size': 14, 'weight': 'bold'},
                linewidths=2, linecolor='black')

    ax2.set_xlabel('Predicted', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Actual', fontsize=12, fontweight='bold')
    ax2.set_title(f'External Validation\n{best_model_name}', fontsize=13, fontweight='bold')

    plt.suptitle(f'Confusion Matrix Comparison - Best Model',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()

    save_path = os.path.join(output_dir, '05_comparison_confusion_matrix.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


# ==========================================
# Step 6: 医学期刊级别对比可视化
# ==========================================

def plot_medical_comparison_figures(df_train, df_external, comparison_df,
                                    trained_models, predictions_dict,
                                    X_test, y_test, output_dir='external_validation'):
    """
    生成医学期刊级别对比可视化图表

    参数:
        df_train: 训练集结果DataFrame
        df_external: 外部验证集结果DataFrame
        comparison_df: 对比结果DataFrame
        trained_models: 训练好的模型字典
        predictions_dict: 外部验证集预测结果字典
        X_test: 外部验证集特征
        y_test: 外部验证集标签
        output_dir: 输出目录
    """
    print("\n" + "=" * 70)
    print("Step 6: 医学期刊级别对比可视化")
    print("=" * 70)

    medical_dir = os.path.join(output_dir, 'medical_comparison')
    os.makedirs(medical_dir, exist_ok=True)

    # 1. DCA对比
    print("\n📊 1. 生成DCA对比图...")
    plot_dca_comparison(trained_models, predictions_dict, y_test, medical_dir)

    # 2. 校准曲线+H-L检验对比
    print("📊 2. 生成校准曲线+H-L检验对比图...")
    plot_calibration_hl_comparison(trained_models, predictions_dict, y_test, medical_dir)

    # 3. Forest Plot对比
    print("📊 3. 生成Forest Plot对比图...")
    plot_forest_comparison(df_train, df_external, medical_dir)

    # 4. 性能下降瀑布图
    print("📊 4. 生成性能下降瀑布图...")
    plot_performance_degradation(comparison_df, medical_dir)

    # 5. 泛化能力雷达图
    print("📊 5. 生成泛化能力雷达图...")
    plot_generalization_radar(df_train, df_external, comparison_df, medical_dir)

    print(f"\n✅ 医学期刊级别对比可视化完成！图表已保存至: {medical_dir}/")


def calculate_net_benefit(y_true, y_proba, threshold):
    """计算指定阈值下的净收益"""
    y_pred = (y_proba >= threshold).astype(int)

    if len(np.unique(y_pred)) == 1:
        if y_pred[0] == 1:
            tp = np.sum(y_true == 1)
            fp = np.sum(y_true == 0)
        else:
            return 0.0
    else:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    n = len(y_true)
    if threshold >= 1.0:
        return 0.0

    net_benefit = (tp / n) - (fp / n) * (threshold / (1 - threshold))
    return net_benefit


def plot_dca_comparison(trained_models, predictions_dict, y_test, output_dir):
    """绘制DCA对比图"""

    # 选择Top 5模型
    top5_models = list(predictions_dict.keys())[:5]

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    thresholds = np.arange(0.01, 0.99, 0.01)
    colors = plt.cm.tab10(np.linspace(0, 1, len(top5_models)))

    for ax_idx, ax in enumerate(axes):
        for model_name, color in zip(top5_models, colors):
            y_proba = predictions_dict[model_name]['y_prob']
            net_benefits = [calculate_net_benefit(y_test, y_proba, t) for t in thresholds]

            ax.plot(thresholds, net_benefits, color=color, linewidth=2.5,
                    label=model_name, alpha=0.8)

        # 参考线
        prevalence = y_test.mean()
        treat_all = [prevalence - (1 - prevalence) * (t / (1 - t)) if t < 1.0 else 0.0
                     for t in thresholds]
        ax.plot(thresholds, treat_all, 'k--', linewidth=2, label='Treat All', alpha=0.7)
        ax.axhline(y=0, color='gray', linestyle=':', linewidth=2, label='Treat None', alpha=0.7)

        ax.set_xlabel('Decision Threshold', fontsize=12, fontweight='bold')
        ax.set_ylabel('Net Benefit', fontsize=12, fontweight='bold')

        if ax_idx == 0:
            ax.set_title('Training Set DCA', fontsize=13, fontweight='bold')
        else:
            ax.set_title('External Validation DCA', fontsize=13, fontweight='bold')

        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim([0, 0.8])

        # 自适应Y轴
        all_nb = net_benefits + treat_all
        y_max = max([nb for nb in all_nb if nb is not None and not np.isnan(nb)]) * 1.1
        ax.set_ylim([-0.05, y_max])

    plt.suptitle('Decision Curve Analysis: Training vs External Validation',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()

    save_path = os.path.join(output_dir, '01_comparison_dca.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


def hosmer_lemeshow_test(y_true, y_prob, n_bins=10):
    """Hosmer-Lemeshow拟合优度检验"""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_prob, bins[:-1]) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)

    observed = np.zeros(n_bins)
    expected = np.zeros(n_bins)
    counts = np.zeros(n_bins)

    for i in range(n_bins):
        mask = bin_indices == i
        counts[i] = mask.sum()
        if counts[i] > 0:
            observed[i] = y_true[mask].sum()
            expected[i] = y_prob[mask].sum()

    mask = counts > 0
    chi2 = np.sum((observed[mask] - expected[mask]) ** 2 /
                  (expected[mask] * (1 - expected[mask] / counts[mask]) + 1e-10))

    p_value = 1 - stats.chi2.cdf(chi2, n_bins - 2)

    return chi2, p_value


def plot_calibration_hl_comparison(trained_models, predictions_dict, y_test, output_dir):
    """绘制校准曲线+H-L检验对比"""
    from sklearn.calibration import calibration_curve

    # 选择Top 6模型
    top6_models = list(predictions_dict.keys())[:6]

    n_models = len(top6_models)
    n_cols = 3
    n_rows = (n_models + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()

    colors = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6', '#1ABC9C']

    for idx, (model_name, color) in enumerate(zip(top6_models, colors)):
        ax = axes[idx]
        y_prob = predictions_dict[model_name]['y_prob']

        # 校准曲线
        fraction_of_positives, mean_predicted_value = calibration_curve(
            y_test, y_prob, n_bins=10, strategy='uniform'
        )

        # H-L检验
        chi2, p_value = hosmer_lemeshow_test(y_test, y_prob, n_bins=10)

        # Brier Score
        brier = brier_score_loss(y_test, y_prob)

        # 绘图
        ax.plot(mean_predicted_value, fraction_of_positives,
                marker='o', color=color, linewidth=2.5, markersize=8,
                label='Calibration Curve', markeredgecolor='black', markeredgewidth=1)
        ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfect', alpha=0.7)

        # 统计信息
        textstr = f'H-L χ²: {chi2:.2f}\np-value: {p_value:.4f}\nBrier: {brier:.4f}'
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        ax.text(0.05, 0.75, textstr, transform=ax.transAxes, fontsize=9,
                verticalalignment='top', bbox=props)

        ax.set_xlabel('Predicted Probability', fontsize=10, fontweight='bold')
        ax.set_ylabel('Observed Proportion', fontsize=10, fontweight='bold')
        ax.set_title(f'{model_name}', fontsize=11, fontweight='bold')
        ax.legend(loc='lower right', fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])

    # 隐藏多余子图
    for idx in range(n_models, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('Calibration Curves with Hosmer-Lemeshow Test - External Validation',
                 fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()

    save_path = os.path.join(output_dir, '02_comparison_calibration_hl.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


def plot_forest_comparison(df_train, df_external, output_dir):
    """绘制Forest Plot对比（训练 | 外部验证）"""

    # 选择共同的模型
    common_models = list(set(df_train.index) & set(df_external.index))
    common_models = sorted(common_models, key=lambda x: df_external.loc[x, 'AUC'], reverse=True)[:10]

    fig, axes = plt.subplots(1, 2, figsize=(16, 10))

    for ax_idx, (ax, df, title) in enumerate(zip(axes,
                                                 [df_train, df_external],
                                                 ['Training Set', 'External Validation'])):
        df_plot = df.loc[common_models].sort_values('AUC', ascending=True)

        y_pos = np.arange(len(df_plot))
        aucs = df_plot['AUC'].values

        # 使用CV结果计算置信区间（如果有）
        if 'CV_AUC_std' in df_plot.columns:
            ci_lower = aucs - 1.96 * df_plot['CV_AUC_std'].values
            ci_upper = aucs + 1.96 * df_plot['CV_AUC_std'].values
        else:
            # 简化版：使用±0.05作为置信区间
            ci_lower = aucs - 0.05
            ci_upper = aucs + 0.05

        ci_lower = np.clip(ci_lower, 0, 1)
        ci_upper = np.clip(ci_upper, 0, 1)

        colors_plot = plt.cm.RdYlGn(aucs)

        for i, (y, auc, lower, upper, color) in enumerate(zip(y_pos, aucs, ci_lower, ci_upper, colors_plot)):
            # 误差条
            ax.plot([lower, upper], [y, y], color='gray', linewidth=2, alpha=0.6)
            # 数据点
            ax.scatter(auc, y, s=200, color=color, edgecolors='black',
                       linewidth=1.5, zorder=3, alpha=0.9)
            # 数值标签
            ax.text(auc + 0.02, y, f'{auc:.3f}', va='center', fontsize=9, fontweight='bold')

        # 参考线
        ax.axvline(x=0.5, color='red', linestyle='--', linewidth=1.5, alpha=0.5)

        ax.set_yticks(y_pos)
        ax.set_yticklabels(df_plot.index, fontsize=9)
        ax.set_xlabel('AUC (95% CI)', fontsize=11, fontweight='bold')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlim([0.5, 1.05])
        ax.grid(True, alpha=0.3, axis='x')

    plt.suptitle('Forest Plot: Model Performance with 95% CI',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()

    save_path = os.path.join(output_dir, '03_comparison_forest_plot.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


def plot_performance_degradation(comparison_df, output_dir):
    """绘制性能下降瀑布图"""

    df_plot = comparison_df.sort_values('AUC_Diff', ascending=False)

    plt.figure(figsize=(14, 10))

    colors = ['#e74c3c' if x > 0.05 else '#f39c12' if x > 0 else '#2ecc71'
              for x in df_plot['AUC_Diff']]

    bars = plt.barh(range(len(df_plot)), df_plot['AUC_Diff'], color=colors, alpha=0.8)

    plt.yticks(range(len(df_plot)), df_plot['Model'], fontsize=10)
    plt.xlabel('AUC Degradation (Training - External)', fontsize=12, fontweight='bold')
    plt.title('Performance Degradation Analysis\n(Positive = Performance Drop, Negative = Performance Improvement)',
              fontsize=14, fontweight='bold')

    # 添加数值标签
    for i, (bar, val) in enumerate(zip(bars, df_plot['AUC_Diff'])):
        width = bar.get_width()
        label_x_pos = width + (0.005 if width > 0 else -0.005)
        ha = 'left' if width > 0 else 'right'
        plt.text(label_x_pos, bar.get_y() + bar.get_height() / 2,
                 f'{val:.4f}', va='center', ha=ha, fontsize=9, fontweight='bold')

    # 参考线
    plt.axvline(x=0, color='black', linestyle='-', linewidth=1.5)
    plt.axvline(x=0.05, color='red', linestyle='--', linewidth=1.5, alpha=0.5,
                label='Overfitting Warning (Δ=0.05)')
    plt.axvline(x=0.10, color='darkred', linestyle='--', linewidth=1.5, alpha=0.7,
                label='Severe Overfitting (Δ=0.10)')

    plt.legend(loc='lower right', fontsize=10)
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()

    save_path = os.path.join(output_dir, '04_performance_degradation.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


def plot_generalization_radar(df_train, df_external, comparison_df, output_dir):
    """绘制泛化能力雷达图"""

    # 选择Top 5模型
    top5_models = comparison_df.head(5)['Model'].tolist()

    metrics = ['AUC', 'Accuracy', 'Sensitivity', 'Specificity', 'F1-Score']

    fig, axes = plt.subplots(2, 3, figsize=(18, 12), subplot_kw=dict(projection='polar'))
    axes = axes.flatten()

    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]  # 闭合

    for idx, model_name in enumerate(top5_models):
        ax = axes[idx]

        # 训练集数据
        train_values = [df_train.loc[model_name, m] for m in metrics]
        train_values += train_values[:1]

        # 外部验证集数据
        external_values = [df_external.loc[model_name, m] for m in metrics]
        external_values += external_values[:1]

        # 绘制
        ax.plot(angles, train_values, 'o-', linewidth=2, label='Training', color='#3498db')
        ax.fill(angles, train_values, alpha=0.25, color='#3498db')

        ax.plot(angles, external_values, 'o-', linewidth=2, label='External', color='#e74c3c')
        ax.fill(angles, external_values, alpha=0.25, color='#e74c3c')

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(metrics, fontsize=9)
        ax.set_ylim([0, 1])
        ax.set_title(f'{model_name}', fontsize=11, fontweight='bold', pad=20)
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
        ax.grid(True)

    # 隐藏最后一个子图
    axes[-1].set_visible(False)

    plt.suptitle('Generalization Ability Radar Chart (Training vs External)',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()

    save_path = os.path.join(output_dir, '05_generalization_radar.pdf')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


# ==========================================
# 执行 Step 4-6（在前3步之后）
# ==========================================

print("\n" + "=" * 70)
print("继续执行 Step 4-6")
print("=" * 70)

# Step 4: 训练 vs 外部验证对比
train_results_file = 'train_results.csv'
external_results_file = 'external_validation/external_results.csv'

comparison_df = compare_train_external_results(
    train_results_path=train_results_file,
    external_results_path=external_results_file,
    output_dir='external_validation'
)

# 重新加载训练集结果用于可视化
df_train = pd.read_csv(train_results_file, index_col=0)

# Step 5: 基础对比可视化
plot_basic_comparison_figures(
    df_train=df_train,
    df_external=df_results,
    comparison_df=comparison_df,
    trained_models=models_dict,
    predictions_dict=predictions,
    X_test=X_external,
    y_test=y_external,
    output_dir='external_validation'
)

# Step 6: 医学期刊级别对比可视化
plot_medical_comparison_figures(
    df_train=df_train,
    df_external=df_results,
    comparison_df=comparison_df,
    trained_models=models_dict,
    predictions_dict=predictions,
    X_test=X_external,
    y_test=y_external,
    output_dir='external_validation'
)

print("\n" + "=" * 70)
print("✅ Step 4-6 完成！")
print("=" * 70)
print("\n生成的文件:")
print("  📄 external_validation/comparison_results.csv")
print("  📁 external_validation/basic_comparison/")
print("     - 01_comparison_roc_curves.pdf")
print("     - 02_comparison_metrics_bar.pdf")


print("     - 03_train_vs_external_scatter.pdf")
print("     - 04_comparison_calibration.pdf")
print("     - 05_comparison_confusion_matrix.pdf")
print("  📁 external_validation/medical_comparison/")
print("     - 01_comparison_dca.pdf")
print("     - 02_comparison_calibration_hl.pdf")
print("     - 03_comparison_forest_plot.pdf")
print("     - 04_performance_degradation.pdf")
print("     - 05_generalization_radar.pdf")

# ==========================================
# SHAP可视化分析模块 - 修复版
# ==========================================

import shap
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
import pandas as pd
import os
from matplotlib.patches import Patch
from matplotlib.colors import Normalize

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False




def prepare_shap_data(trained_models, X_train, X_test, y_test, results,
                      feature_names, output_dir='shap_analysis', max_samples=150):
    print("\n" + "=" * 70)
    print("🔍 SHAP 可解释性分析 - 数据准备")
    print("=" * 70)

    os.makedirs(output_dir, exist_ok=True)

    df_results = pd.DataFrame(results).T
    best_model_name = df_results['AUC'].idxmax()
    best_model = trained_models[best_model_name]

    print(f"\n✅ 选择最优模型: {best_model_name}")
    print(f"   - AUC: {df_results.loc[best_model_name, 'AUC']:.4f}")
    print(f"   - F1-Score: {df_results.loc[best_model_name, 'F1-Score']:.4f}")

    if X_test.shape[0] > max_samples:
        idx = np.random.choice(X_test.shape[0], max_samples, replace=False)
        X_sample = X_test[idx]
        y_sample = y_test.iloc[idx] if isinstance(y_test, pd.Series) else y_test[idx]
    else:
        X_sample = X_test
        y_sample = y_test

    print(f"\n📊 SHAP 计算样本数: {X_sample.shape[0]}")
    print("\n🧠 正在创建 SHAP 解释器...")

    shap_values = None
    base_value = None
    explainer = None

    try:
        # 清洗 XGBoost base_score
        if hasattr(best_model, "get_booster"):
            booster = best_model.get_booster()
            bs = booster.attr("base_score")
            if isinstance(bs, str) and bs.startswith("[") and bs.endswith("]"):
                bs_clean = bs.strip("[] ")
                try:
                    booster.set_attr(base_score=str(float(bs_clean)))
                except ValueError:
                    pass

        try:
            explainer = shap.TreeExplainer(best_model)
            shap_values_raw = explainer.shap_values(X_sample)
            print("   -> 使用 TreeExplainer (适用于树模型)")
            base_value = explainer.expected_value
        except Exception:
            masker = shap.maskers.Independent(X_sample)
            explainer = shap.Explainer(lambda x: best_model.predict_proba(x)[:, 1],
                                       masker, feature_names=feature_names)
            explanation = explainer(X_sample)
            shap_values_raw = explanation.values
            base_value = explanation.base_values
            print("   -> 回退 shap.Explainer + predict_proba")

        # 处理 shap_values_raw
        if isinstance(shap_values_raw, list):
            shap_values = shap_values_raw[1]
        elif isinstance(shap_values_raw, np.ndarray):
            if shap_values_raw.ndim == 3:
                shap_values = shap_values_raw[:, :, 1]
            elif shap_values_raw.ndim == 2:
                shap_values = shap_values_raw
            else:
                raise ValueError(f"不支持的SHAP值维度: {shap_values_raw.ndim}")
        else:
            raise ValueError(f"不支持的SHAP值类型: {type(shap_values_raw)}")

        # 处理 base_value
        if isinstance(base_value, (list, np.ndarray)):
            base_value = base_value[1] if len(base_value) > 1 else base_value[0]

        print(f"✅ SHAP 值计算完成，最终形状: {shap_values.shape}")
        print(f"   - 预期形状: ({X_sample.shape[0]}, {len(feature_names)})")
        print(f"   - 基准值 (Base Value): {base_value:.4f}")

        if shap_values.shape != (X_sample.shape[0], len(feature_names)):
            raise ValueError(
                f"SHAP值形状不匹配！实际: {shap_values.shape}, 预期: ({X_sample.shape[0]}, {len(feature_names)})"
            )

    except Exception as e:
        print(f"❌ SHAP 计算失败: {e}")
        import traceback
        traceback.print_exc()
        return None

    return {
        'best_model': best_model,
        'best_model_name': best_model_name,
        'explainer': explainer,
        'shap_values': shap_values,
        'base_value': base_value,
        'X_sample': X_sample,
        'y_sample': y_sample,
        'feature_names': feature_names,
        'y_pred_proba': best_model.predict_proba(X_sample)[:, 1]
    }

# ==========================================
# 2. SHAP条形图（特征重要性）
# ==========================================

def plot_shap_bar_importance(shap_data, output_dir='shap_analysis'):
    """绘制SHAP特征重要性条形图"""
    print("\n📊 1. 生成 SHAP 特征重要性条形图...")

    shap_values = shap_data['shap_values']
    feature_names = shap_data['feature_names']
    X_sample = shap_data['X_sample']

    # ⭐ 再次验证形状
    print(f"   -> SHAP值形状: {shap_values.shape}")
    print(f"   -> X_sample形状: {X_sample.shape}")
    print(f"   -> 特征名称数量: {len(feature_names)}")

    plt.figure(figsize=(12, 8))

    # 将X_sample转为DataFrame以确保特征名正确
    X_sample_df = pd.DataFrame(X_sample, columns=feature_names)

    shap.summary_plot(
        shap_values,
        X_sample_df,
        plot_type="bar",
        show=False,
        max_display=20
    )

    plt.title(f"SHAP 特征重要性 ({shap_data['best_model_name']})",
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel("平均绝对 SHAP 值 (对预测结果的影响程度)", fontsize=12)
    plt.tight_layout()

    save_path = f'{output_dir}/01_shap_feature_importance_bar.pdf'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")
# ==========================================
# 3. SHAP蜂群图（Summary Plot）
# ==========================================

def plot_shap_summary_beeswarm(shap_data, output_dir='shap_analysis'):
    """绘制SHAP蜂群图"""
    print("\n📊 2. 生成 SHAP Summary Plot (蜂群图)...")

    shap_values = shap_data['shap_values']
    feature_names = shap_data['feature_names']
    X_sample = shap_data['X_sample']

    plt.figure(figsize=(12, 10))
    shap.summary_plot(
        shap_values,
        X_sample,
        feature_names=feature_names,
        plot_type="dot",
        max_display=20,
        show=False
    )

    plt.title('SHAP 特征影响分析 (蜂群图)', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('SHAP value (对模型预测的影响: 右=增加风险, 左=降低风险)', fontsize=12)
    plt.tight_layout()

    save_path = f'{output_dir}/02_shap_summary_beeswarm.pdf'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


# ==========================================
# 4. SHAP统计报告
# ==========================================

def generate_shap_statistics(shap_data, output_dir='shap_analysis'):
    """生成SHAP统计报告"""
    print("\n📊 3. 生成 SHAP 统计分析报告...")

    shap_values = shap_data['shap_values']
    feature_names = shap_data['feature_names']

    # 计算统计指标
    shap_stats_df = pd.DataFrame({
        'Feature': feature_names,
        'Mean_|SHAP|': np.abs(shap_values).mean(0),
        'Std_SHAP': shap_values.std(0),
        'Max_|SHAP|': np.abs(shap_values).max(0),
        'Min_|SHAP|': np.abs(shap_values).min(0)
    }).sort_values('Mean_|SHAP|', ascending=False)

    # 保存CSV
    stats_csv_path = f'{output_dir}/shap_statistics_report.csv'
    shap_stats_df.to_csv(stats_csv_path, index=False, encoding='utf-8-sig')

    print(f"   ✅ 已保存: {stats_csv_path}")
    print("\n📋 Top 15 特征统计详情:")
    print(shap_stats_df.head(15).to_string(index=False))

    return shap_stats_df


# ==========================================
# 5. SHAP瀑布图（典型样本）
# ==========================================

def plot_shap_waterfall_samples(shap_data, output_dir='shap_analysis', n_samples=5):
    """绘制典型样本的瀑布图"""
    print(f"\n📊 4. 生成 SHAP 瀑布图 (典型 {n_samples} 样本)...")

    shap_values = shap_data['shap_values']
    base_value = shap_data['base_value']
    X_sample = shap_data['X_sample']
    feature_names = shap_data['feature_names']
    y_pred_proba = shap_data['y_pred_proba']
    y_sample = shap_data['y_sample']

    # 构建Explanation对象
    shap_exp = shap.Explanation(
        values=shap_values,
        base_values=base_value,
        data=X_sample,
        feature_names=feature_names
    )

    # 选择典型样本
    sample_indices = []

    # 1. 高风险样本
    high_risk_idx = np.argmax(y_pred_proba)
    sample_indices.append((high_risk_idx, 'Highest_Risk'))

    # 2. 低风险样本
    low_risk_idx = np.argmin(y_pred_proba)
    sample_indices.append((low_risk_idx, 'Lowest_Risk'))

    # 3. 边界样本
    borderline_idx = np.argmin(np.abs(y_pred_proba - 0.5))
    sample_indices.append((borderline_idx, 'Borderline'))

    # 4. 随机样本
    random_indices = np.random.choice(len(X_sample), min(n_samples - 3, len(X_sample)), replace=False)
    for i, idx in enumerate(random_indices):
        sample_indices.append((idx, f'Random_{i+1}'))

    # 绘制瀑布图
    for sample_idx, label in sample_indices[:n_samples]:
        plt.figure(figsize=(10, 8))

        shap.plots.waterfall(
            shap_exp[sample_idx],
            max_display=15,
            show=False
        )

        pred_prob = y_pred_proba[sample_idx]
        actual_label = 'Positive' if y_sample.iloc[sample_idx] == 1 else 'Negative' if isinstance(y_sample, pd.Series) else 'Unknown'
        pred_label = 'Positive' if pred_prob > 0.5 else 'Negative'

        plt.title(
            f'样本瀑布图: {label}\n'
            f'预测概率: {pred_prob:.3f} | 预测类别: {pred_label} | 真实类别: {actual_label}',
            fontsize=14, fontweight='bold', pad=20
        )
        plt.tight_layout()

        save_path = f'{output_dir}/03_shap_waterfall_{label}.pdf'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✅ 已保存: {save_path}")


# ==========================================
# 6. SHAP依赖图（Top特征）
# ==========================================

def plot_shap_dependence(shap_data, output_dir='shap_analysis', top_n=4):
    """绘制SHAP依赖图"""
    print(f"\n📊 5. 生成 SHAP 依赖图 (Top {top_n} 特征)...")

    shap_values = shap_data['shap_values']
    X_sample = shap_data['X_sample']
    feature_names = shap_data['feature_names']

    # 计算特征重要性
    shap_importance = np.abs(shap_values).mean(0)
    top_features_idx = np.argsort(shap_importance)[::-1][:top_n]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    for i, feat_idx in enumerate(top_features_idx):
        feature_name = feature_names[feat_idx]

        plt.sca(axes[i])
        shap.dependence_plot(
            feat_idx,
            shap_values,
            X_sample,
            feature_names=feature_names,
            ax=axes[i],
            show=False,
            interaction_index='auto',
            alpha=0.8
        )
        axes[i].set_title(f'{feature_name} 的依赖图', fontsize=12, fontweight='bold')
        axes[i].set_ylabel('SHAP value', fontsize=11)
        axes[i].grid(True, alpha=0.3)

    plt.suptitle(f'Top {top_n} 特征的 SHAP 依赖图分析', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()

    save_path = f'{output_dir}/04_shap_dependence_top{top_n}.pdf'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


# ==========================================
# 7. SHAP Force Plot（静态多样本）
# ==========================================

def plot_shap_force_plots(shap_data, output_dir='shap_analysis', n_samples=3):
    """绘制Force Plot（静态图片）"""
    print(f"\n📊 6. 生成静态 Force Plot ({n_samples} 样本)...")

    shap_values = shap_data['shap_values']
    base_value = shap_data['base_value']
    X_sample = shap_data['X_sample']
    feature_names = shap_data['feature_names']
    y_pred_proba = shap_data['y_pred_proba']

    # 保留2位小数
    shap_values_rounded = np.round(shap_values, 2)
    X_values_rounded = np.round(X_sample, 2)

    # 随机选择样本
    plot_indices = np.random.choice(len(X_sample), min(n_samples, len(X_sample)), replace=False)

    for idx in plot_indices:
        plt.figure(figsize=(20, 5))

        shap.force_plot(
            base_value,
            shap_values_rounded[idx],
            X_values_rounded[idx],
            feature_names=feature_names,
            matplotlib=True,
            show=False,
            text_rotation=15
        )

        current_prob = y_pred_proba[idx]
        plt.title(f'Force Plot (Sample {idx}, 预测概率={current_prob:.3f})',
                  fontsize=14, fontweight='bold', y=1.7)

        save_path = f'{output_dir}/05_shap_force_plot_sample_{idx}.pdf'
        plt.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0.5)
        plt.close()
        print(f"   ✅ 已保存: {save_path}")


# ==========================================
# 8. SHAP Decision Plot（决策路径）
# ==========================================

def plot_shap_decision(shap_data, output_dir='shap_analysis', n_samples=100):
    """绘制SHAP决策路径图"""
    print(f"\n📊 7. 生成 SHAP 决策路径图 ({n_samples} 样本)...")

    shap_values = shap_data['shap_values']
    base_value = shap_data['base_value']
    X_sample = shap_data['X_sample']
    feature_names = shap_data['feature_names']

    n_decision_samples = min(n_samples, len(X_sample))
    decision_indices = np.random.choice(len(X_sample), n_decision_samples, replace=False)

    plt.figure(figsize=(12, 8))

    # 将X_sample转为DataFrame以显示特征名
    X_sample_df = pd.DataFrame(X_sample, columns=feature_names)

    shap.decision_plot(
        base_value,
        shap_values[decision_indices],
        X_sample_df.iloc[decision_indices],
        feature_names=feature_names,
        show=False,
        link='logit'
    )

    plt.title(f'SHAP 决策路径分析 (随机 {n_decision_samples} 个样本)',
              fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()

    save_path = f'{output_dir}/06_shap_decision_plot.pdf'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


# ==========================================
# 9. SHAP热力图
# ==========================================

def plot_shap_heatmap(shap_data, output_dir='shap_analysis'):
    """绘制SHAP值热力图"""
    print("\n📊 8. 生成 SHAP 值热力图...")

    shap_values = shap_data['shap_values']
    X_sample = shap_data['X_sample']
    feature_names = shap_data['feature_names']
    y_sample = shap_data['y_sample']

    # 计算特征重要性
    shap_importance = np.abs(shap_values).mean(0)
    top_features_idx = np.argsort(shap_importance)[::-1][:15]

    # 按SHAP总和排序样本
    shap_sum = shap_values.sum(1)
    sorted_indices = np.argsort(shap_sum)

    # 选择前50个样本
    n_samples_heatmap = min(50, len(sorted_indices))
    selected_indices = sorted_indices[-n_samples_heatmap:]

    plt.figure(figsize=(15, 10))

    # 准备热力图数据
    shap_heatmap_data = shap_values[selected_indices][:, top_features_idx].T

    # 绘制热力图
    sns.heatmap(shap_heatmap_data,
                yticklabels=[feature_names[i] for i in top_features_idx],
                xticklabels=[],
                cmap='RdBu_r',
                center=0,
                cbar_kws={'label': 'SHAP Value'})

    # 添加底部样本类别标记
    y_sample_array = y_sample.values if isinstance(y_sample, pd.Series) else y_sample
    actual_labels_heatmap = y_sample_array[selected_indices]

    for i, label in enumerate(actual_labels_heatmap):
        color = 'red' if label == 1 else 'blue'
        plt.axvline(x=i + 0.5, color=color, alpha=0.6, linewidth=3, ymin=0, ymax=0.02)

    plt.xlabel('样本 (按 SHAP 总和排序)', fontsize=12)
    plt.ylabel('特征 (按重要性排序)', fontsize=12)
    plt.title('SHAP 值热力图 (红色=增加风险, 蓝色=降低风险)',
              fontsize=16, fontweight='bold', pad=20)

    # 添加图例
    legend_elements = [
        Patch(facecolor='red', alpha=0.5, label='真实标签: 阳性'),
        Patch(facecolor='blue', alpha=0.5, label='真实标签: 阴性')
    ]
    plt.legend(handles=legend_elements, loc='upper right')

    plt.tight_layout()

    save_path = f'{output_dir}/07_shap_heatmap.pdf'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


# ==========================================
# 10. SHAP综合看板
# ==========================================

def plot_shap_comprehensive_dashboard(shap_data, output_dir='shap_analysis'):
    """绘制SHAP综合可视化看板"""
    print("\n📊 9. 生成 SHAP 综合看板...")

    shap_values = shap_data['shap_values']
    X_sample = shap_data['X_sample']
    feature_names = shap_data['feature_names']
    y_sample = shap_data['y_sample']

    # 计算特征重要性
    shap_importance = np.abs(shap_values).mean(0)
    importance_ranking = np.argsort(shap_importance)[::-1]

    # 创建大画布
    fig = plt.figure(figsize=(24, 16))
    gs = gridspec.GridSpec(3, 4, figure=fig, wspace=0.4, hspace=0.4)

    # --- 左侧: Summary Plot (3行 x 2列) ---
    ax_main = fig.add_subplot(gs[:, :2])

    # 准备数据
    top_20_idx = importance_ranking[:20]
    top_20_names = [feature_names[i] for i in top_20_idx]
    top_20_importance = shap_importance[top_20_idx]

    # 设置Y轴
    ax_main.set_yticks(range(20))
    ax_main.set_yticklabels(top_20_names[::-1], fontsize=10)

    # 顶部添加平均重要性条形图
    ax_top = ax_main.twiny()
    ax_top.barh(range(20), top_20_importance[::-1],
                color="lightgray", alpha=0.5, height=0.7)
    ax_top.set_xlabel("平均 |SHAP 值|", fontsize=12, fontweight='bold')

    # 绘制蜂群散点图
    cmap_summary = plt.get_cmap("viridis")
    scatter_plots = []

    for i, feat_idx in enumerate(top_20_idx[::-1]):
        shap_vals_feat = shap_values[:, feat_idx]
        feature_vals_feat = X_sample[:, feat_idx]

        y_jitter = np.random.normal(0, 0.08, len(shap_vals_feat))

        sc = ax_main.scatter(shap_vals_feat, i + y_jitter,
                             c=feature_vals_feat, cmap=cmap_summary,
                             s=20, alpha=0.8, edgecolor='none')
        if i == 0:
            scatter_plots.append(sc)

    ax_main.set_xlabel("SHAP 值 (对预测结果的影响)", fontsize=12, fontweight='bold')
    ax_main.grid(True, axis='x', linestyle='--', alpha=0.4)
    ax_main.set_title('全局特征重要性与分布 (Summary Plot)', fontsize=14, fontweight='bold')

    # 添加颜色条
    cax_summary = fig.add_axes([0.15, 0.08, 0.25, 0.015])
    cbar_summary = fig.colorbar(scatter_plots[0], cax=cax_summary, orientation='horizontal')
    cbar_summary.set_label('特征值 (颜色: 低 -> 高)', fontsize=10)

    # --- 右侧: Dependence Plots (3行 x 2列) ---
    top_6_features_idx = importance_ranking[:6]
    axes_scatter = []

    for i in range(3):
        for j in range(2):
            axes_scatter.append(fig.add_subplot(gs[i, j + 2]))

    y_sample_array = y_sample.values if isinstance(y_sample, pd.Series) else y_sample

    for i, feat_idx in enumerate(top_6_features_idx):
        ax = axes_scatter[i]
        feature_name = feature_names[feat_idx]

        x_data = X_sample[:, feat_idx]
        y_data = shap_values[:, feat_idx]

        scatter = ax.scatter(x_data, y_data, c=y_sample_array,
                             cmap='coolwarm', s=40, alpha=0.7,
                             edgecolor='k', linewidth=0.3)

        # 添加趋势线
        if len(x_data) > 1:
            try:
                z = np.polyfit(x_data, y_data, 1)
                p = np.poly1d(z)
                x_line = np.linspace(x_data.min(), x_data.max(), 100)
                ax.plot(x_line, p(x_line), 'k--', linewidth=1.5, alpha=0.8)

                corr = np.corrcoef(x_data, y_data)[0, 1]
                ax.text(0.05, 0.95, f'r = {corr:.2f}', transform=ax.transAxes,
                        fontsize=10, verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            except:
                pass

        ax.set_xlabel(feature_name, fontsize=11)
        ax.set_ylabel('SHAP Value', fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_title(f'Top {i+1}: {feature_name}', fontsize=11, fontweight='bold')

    # 添加右侧颜色条
    cax_dep = fig.add_axes([0.92, 0.4, 0.01, 0.2])
    cbar_dep = fig.colorbar(scatter, cax=cax_dep)
    cbar_dep.set_label('真实标签 (蓝=阴性, 红=阳性)', fontsize=10)
    cbar_dep.set_ticks([0, 1])

    # 总标题
    plt.suptitle(f'SHAP 综合分析看板: {shap_data["best_model_name"]}',
                 fontsize=20, y=0.96, fontweight='bold')

    save_path = f'{output_dir}/08_shap_comprehensive_dashboard.pdf'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")


# ==========================================
# 11. LIME对比分析（可选）
# ==========================================

def plot_lime_analysis(shap_data, X_train, output_dir='shap_analysis'):
    """LIME局部解释性分析（与SHAP对比）"""
    print("\n📊 10. 生成 LIME 分析...")

    best_model = shap_data['best_model']
    X_sample = shap_data['X_sample']
    y_sample = shap_data['y_sample']
    feature_names = shap_data['feature_names']
    y_pred_proba = shap_data['y_pred_proba']
    shap_values = shap_data['shap_values']

    # 初始化LIME Explainer
    X_train_sample = X_train[np.random.choice(X_train.shape[0], min(5000, X_train.shape[0]), replace=False)]

    explainer_lime = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train_sample,
        feature_names=feature_names,
        class_names=['Low Risk', 'High Risk'],
        mode='classification',
        discretize_continuous=True,
        random_state=42
    )

    # 选择高风险样本
    high_risk_idx = np.argmax(y_pred_proba)
    sample_instance = X_sample[high_risk_idx]
    sample_prob = y_pred_proba[high_risk_idx]

    # LIME解释
    exp_lime = explainer_lime.explain_instance(
        sample_instance,
        best_model.predict_proba,
        num_features=10
    )

    # 绘制LIME图
    exp_list = exp_lime.as_list()
    features = [x[0] for x in exp_list]
    weights = [x[1] for x in exp_list]
    colors = ['#d62728' if w > 0 else '#1f77b4' for w in weights]

    plt.figure(figsize=(10, 6))
    bars = plt.barh(range(len(weights)), weights, color=colors, alpha=0.8, height=0.6)

    plt.yticks(range(len(features)), features, fontsize=11)
    plt.gca().invert_yaxis()

    for bar in bars:
        width = bar.get_width()
        label_x_pos = width + (max(weights) * 0.02 if width > 0 else min(weights) * 0.02)
        plt.text(label_x_pos, bar.get_y() + bar.get_height() / 2,
                 f'{width:.3f}', va='center', fontsize=10)

    y_sample_array = y_sample.values if isinstance(y_sample, pd.Series) else y_sample
    true_label = y_sample_array[high_risk_idx]
    label_str = "High Risk (1)" if true_label == 1 else "Low Risk (0)"
    pred_class = "High Risk" if sample_prob >= 0.5 else "Low Risk"

    plt.title(f'LIME Local Explanation (Sample {high_risk_idx})\n'
              f'Prediction: {sample_prob:.3f} ({pred_class}) | True Label: {label_str}',
              fontsize=14, fontweight='bold')
    plt.xlabel('Feature Contribution (Weight)', fontsize=12)
    plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
    plt.grid(axis='x', linestyle=':', alpha=0.5)

    legend_elements = [
        Patch(facecolor='#d62728', label='Increases Risk (Positive)'),
        Patch(facecolor='#1f77b4', label='Decreases Risk (Negative)')
    ]
    plt.legend(handles=legend_elements, loc='lower right')

    plt.tight_layout()
    save_path = f'{output_dir}/09_lime_high_risk_sample.pdf'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ 已保存: {save_path}")

    # LIME vs SHAP对比
    shap_vals_sample = shap_values[high_risk_idx]
    comparison_data = []

    for lime_feat_name, lime_weight in exp_list:
        for raw_feat in feature_names:
            if raw_feat in lime_feat_name:
                feat_idx = feature_names.index(raw_feat)
                shap_val = shap_vals_sample[feat_idx]
                comparison_data.append({
                    'Feature': raw_feat,
                    'LIME_Weight': lime_weight,
                    'SHAP_Value': shap_val
                })
                break

    if comparison_data:
        comp_df = pd.DataFrame(comparison_data)

        fig, ax = plt.subplots(figsize=(12, 7))
        y_pos = np.arange(len(comp_df))
        height = 0.35

        ax.barh(y_pos + height/2, comp_df['LIME_Weight'], height,
                label='LIME Weight', color='#2ca02c', alpha=0.8)
        ax.barh(y_pos - height/2, comp_df['SHAP_Value'], height,
                label='SHAP Value', color='#ff7f0e', alpha=0.8)

        ax.set_yticks(y_pos)
        ax.set_yticklabels(comp_df['Feature'], fontsize=10)
        ax.invert_yaxis()

        ax.set_xlabel('Contribution Value', fontsize=12)
        ax.set_title(f'LIME vs SHAP Comparison (Sample {high_risk_idx})',
                     fontsize=14, fontweight='bold')
        ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
        ax.legend()

        correlation = comp_df['LIME_Weight'].corr(comp_df['SHAP_Value'])
        plt.figtext(0.15, 0.02, f"Correlation: r = {correlation:.2f}",
                    fontsize=10, bbox=dict(facecolor='white', alpha=0.8))

        plt.tight_layout()
        save_path = f'{output_dir}/10_lime_vs_shap_comparison.pdf'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✅ 已保存: {save_path}")


# ==========================================
# 11：生成交互式SHAP Force Plot (HTML)
# ==========================================

def plot_shap_force_interactive_html(shap_data, output_dir='shap_analysis'):
    """
    生成交互式SHAP Force Plot (HTML格式)
    这种图将所有样本堆叠起来，可以在浏览器中交互查看
    """
    print("\n📊 11. 生成交互式 SHAP Force Plot (HTML)...")

    shap_values = shap_data['shap_values']
    base_value = shap_data['base_value']
    X_sample = shap_data['X_sample']
    feature_names = shap_data['feature_names']

    try:
        # 将X_sample转为DataFrame以显示特征名
        X_sample_df = pd.DataFrame(X_sample, columns=feature_names)

        # 生成交互式对象 (注意：不加 matplotlib=True)
        # 这种图将所有样本堆叠起来，横轴是样本（可以按相似度聚类），纵轴是SHAP值
        interactive_plot = shap.force_plot(
            base_value,
            shap_values,
            X_sample_df,
            feature_names=feature_names
        )

        # 保存为HTML文件
        html_path = f'{output_dir}/11_shap_force_plot_interactive.html'
        shap.save_html(html_path, interactive_plot)

        print(f"✅ 交互式 HTML 已保存: {html_path}")
        print("   -> 请在浏览器中打开此文件，体验动态交互功能")
        print("   -> 功能包括: 按特征排序、查看样本聚类、悬停查看详情等")

    except Exception as e:
        print(f"⚠️ 保存交互式 HTML 失败 (可能是依赖包问题): {e}")
        print("   -> 建议: 确保已安装最新版本的 shap 库")
        print("   -> 可以尝试: pip install --upgrade shap")

# ==========================================
# 主函数：执行完整SHAP分析流程
# ==========================================

def run_complete_shap_analysis(trained_models, X_train, X_test, y_test,
                               results, feature_names, output_dir='shap_analysis'):
    """
    执行完整的SHAP可解释性分析流程

    参数:
        trained_models: 训练好的模型字典
        X_train: 训练集
        X_test: 测试集
        y_test: 测试标签
        results: 模型结果字典
        feature_names: 特征名称列表
        output_dir: 输出目录
    """
    print("\n" + "=" * 70)
    print("🔍 开始完整SHAP可解释性分析")
    print("=" * 70)

    # 1. 准备SHAP数据
    shap_data = prepare_shap_data(
        trained_models, X_train, X_test, y_test,
        results, feature_names, output_dir, max_samples=150
    )

    if shap_data is None:
        print("❌ SHAP数据准备失败，终止分析")
        return

    # 2. 生成各类可视化
    plot_shap_bar_importance(shap_data, output_dir)
    plot_shap_summary_beeswarm(shap_data, output_dir)
    shap_stats_df = generate_shap_statistics(shap_data, output_dir)
    plot_shap_waterfall_samples(shap_data, output_dir, n_samples=5)
    plot_shap_dependence(shap_data, output_dir, top_n=4)
    plot_shap_force_plots(shap_data, output_dir, n_samples=3)
    plot_shap_decision(shap_data, output_dir, n_samples=20)
    plot_shap_heatmap(shap_data, output_dir)
    plot_shap_comprehensive_dashboard(shap_data, output_dir)
    plot_shap_force_interactive_html(shap_data, output_dir)

    # 3. LIME对比分析（可选）
    try:
        plot_lime_analysis(shap_data, X_train, output_dir)
    except Exception as e:
        print(f"⚠️ LIME分析失败（可跳过）: {e}")

    print("\n" + "=" * 70)
    print("🎉 所有SHAP分析完成！")
    print("=" * 70)
    print(f"\n📁 结果保存在目录: {output_dir}/")
    print("\n生成的图表:")
    print("  01. SHAP特征重要性条形图")
    print("  02. SHAP蜂群图（Summary Plot）")
    print("  03. SHAP瀑布图（典型样本）")
    print("  04. SHAP依赖图（Top特征）")
    print("  05. SHAP Force Plot（静态）")
    print("  06. SHAP决策路径图")
    print("  07. SHAP热力图")
    print("  08. SHAP综合看板")
    print("  09. LIME局部解释")
    print("  10. LIME vs SHAP对比")
    print("  11. 交互式SHAP Force Plot (HTML)")
    print("  + shap_statistics_report.csv（统计数据）")


# ==========================================
# 在主流程末尾调用
# ==========================================
# 执行完整SHAP分析
# python
# 确保 SHAP 分析可运行的变量映射
all_trained = models_dict
X_train_scaled = X_external
X_test_scaled = X_external
y_test = y_external
all_results = results

run_complete_shap_analysis(
    all_trained,        # 所有训练好的模型
    X_train_scaled,     # 训练集（此处复用外部验证集）
    X_test_scaled,      # 测试集（此处复用外部验证集）
    y_test,             # 测试标签
    all_results,        # 所有结果字典
    feature_names,      # 特征名称列表
    output_dir='shap_analysis'  # 输出目录
)

print("\n" + "=" * 70)
print("所有分析和可视化已完成！")
print("=" * 70)


🚀 外部验证集评估流程 - 前3步

Step 1: 模型和配置加载
✅ 已加载标准化器: saved_models\scaler1.pkl
✅ 已加载特征名称: saved_models\feature_names1.pkl
   特征数量: 20

📦 开始加载模型...
   ℹ️ 已排除: Voting_Hard, Voting Hard
  ✅ AdaBoost                            加载成功
  ✅ Decision Tree                       加载成功
  ✅ Extra Trees                         加载成功
  ✅ KNN                                 加载成功
  ✅ LightGBM                            加载成功
  ✅ LightGBM Optimized                  加载成功
  ✅ Logistic Regression                 加载成功
  ✅ Naive Bayes                         加载成功
  ✅ Neural Network                      加载成功
  ✅ Random Forest                       加载成功
  ✅ Random Forest Optimized             加载成功
  ✅ SVM                                 加载成功
  ✅ Stacking                            加载成功
  ✅ Voting Soft                         加载成功
  ✅ XGBoost                             加载成功
  ✅ XGBoost Optimized                   加载成功
  ⚠️ feature names                       不支持概率预测，已跳过
  ⚠️ scaler                              不支持概率预测，已跳

MERG NOT subset; don't know how to subset; dropped
MERG NOT subset; don't know how to subset; dropped


   ✅ 已保存: external_validation\basic_comparison\01_comparison_roc_curves.pdf
📊 2. 生成指标对比柱状图...


MERG NOT subset; don't know how to subset; dropped
MERG NOT subset; don't know how to subset; dropped


   ✅ 已保存: external_validation\basic_comparison\02_comparison_metrics_bar.pdf
📊 3. 生成泛化能力散点图...


MERG NOT subset; don't know how to subset; dropped
MERG NOT subset; don't know how to subset; dropped


... (22 outputs left out)

   ✅ 已保存: shap_analysis/03_shap_waterfall_Lowest_Risk.pdf


MERG NOT subset; don't know how to subset; dropped
MERG NOT subset; don't know how to subset; dropped


   ✅ 已保存: shap_analysis/03_shap_waterfall_Borderline.pdf


MERG NOT subset; don't know how to subset; dropped
